<a href="https://colab.research.google.com/github/Jonchyk/Datamgmt/blob/main/Capstone_Workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Set Up

In [1]:
#---------------------------SETUP----------------------------------
#get useful libraries
import time, os, sys, re #basics
import zipfile, json, datetime, string   #string for annotating points in scatter
import numpy as np #basic math
# Install tabula-py using pip
!pip install tabula-py
# Import the correct module
from tabula import read_pdf
import statsmodels.api as sm
from statsmodels.formula.api import ols
!pip install pyreadstat
import plotly.express as px
from statsmodels.stats.multicomp import pairwise_tukeyhsd


import matplotlib.pyplot as plt #import pylab as plt #apparently discouraged now:
 #https://stackoverflow.com/questions/11469336/what-is-the-difference-between-pylab-and-pyplot
 #https://www.tutorialspoint.com/matplotlib/matplotlib_pylab_module.htm

import pandas as pd
import pandas_datareader as pdr
from pandas_datareader import wb
from pandas.io.formats.style import Styler
#s4 = Styler(df4, uuid_len=0, cell_ids=False)

import urllib  #weird, guess need to have os and pandas imported for this to work  %TODO/LATER ditch it, its weird anyway, just use wget/curl

from google.colab import files

#import webbrowser

import seaborn as sns

from google.colab import data_table
data_table.enable_dataframe_formatter() #this enables spreadsheet view upon calling dataframe (without() )

#many tricks how to extend notebook functionality
#https://coderzcolumn.com/tutorials/python/list-of-useful-magic-commands-in-jupyter-notebook-lab
#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#MAGICS and THEMES/STYLES: important! does affect not just shading/colors, but also fonts, spacing, etc
#(even if you only select default (v not selecting anything) [but does seem to work better if you do make explicit sleections])

###magics: https://ipython.readthedocs.io/en/stable/interactive/magics.html
#most essential setup for vis: it does affect vis! careful!! stick with inline, maybe notebook; others mostly for non-notebook, eg spyder environ
#https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html recomends *inline*!
#show current one:
#%matplotlib
#%matplotlib --list
#interactive plots:
#%matplotlib notebook
#static images of your plot:
%matplotlib inline
#may play with this one and other magics (btw default is probably agg)
#%matplotlib nbagg
##https://www.marktechpost.com/2023/10/20/6-magic-commands-for-jupyter-notebooks-in-python-data-science/
#%%latex
#%ai
#%run
#%writefile
#%history -n

###themes/styles: https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html
#https://jakevdp.github.io/PythonDataScienceHandbook/04.11-settings-and-stylesheets.html
#https://matplotlib.org/stable/tutorials/introductory/customizing.html
#here more about art and style than under the hood functionality as with magics, explore and experiment
#many may find 'default' or seaborn ones more pleasing; my fav 'classic' is back from 90s ;)
#plt.style.available #list available styles :) may install more
#plt.style.use('default') # more delicate subtle than classic
plt.style.use('classic')  #  'seaborn-whitegrid' 'seaborn-white' 'seaborn-poster'
# btw: magics v theme/style sequence matters, eg if i specify classic style before inline magic, i wouldnt get grey bounding box im getting

#sometimes have to install library which you get from https://pypi.org/
#!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 18.6 MB/s eta 0:00:00


#Datasets



##Ministry of Science and Education Report - Population and Ethnic Demographics by Region
Report accessed here, table with population information summarized and rebuilt using Tabula. All data is from 2022

 https://stat.gov.kg/media/publicationarchive/fa3392ca-c76e-456f-badd-7a324ff5204d.pdf

In [2]:
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn" -O moesreport.pdf


--2025-04-09 23:21:18--  https://drive.google.com/uc?export=download&id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn
Resolving drive.google.com (drive.google.com)... 74.125.26.139, 74.125.26.100, 74.125.26.102, ...
Connecting to drive.google.com (drive.google.com)|74.125.26.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn&export=download [following]
--2025-04-09 23:21:18--  https://drive.usercontent.google.com/download?id=1fTVSC1y_1MSfCS44n7hR-Z7eU-vLtLTn&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.98.132, 2607:f8b0:400c:c1a::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.98.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1484754 (1.4M) [application/octet-stream]
Saving to: ‘moesreport.pdf’

moesreport.pdf      100%[===================>]   1.42M  --.-KB/s    

## Early Grade Reading Assessment Data

Early Grade Reading Assessment: received with permission from employer, RTI.
Dataset can be accessed at: data.usaid.gov

This is a dataset from the Okuu Keremet! Program's 2021 Baseline study. It has a wealth of information, ranging from G2 teacher, head teacher, and librarian interviews, classroom observations, math results, and early grade reading results. For the purpose of my capstone study, I will be using this dataset, cleaned and targeting specifically the EGRA subtasks: Oral Reading, Oral Reading Comprehension, Invented Word Reading,  Silent Reading Comprehension, and listening comprehension.


In [3]:
# let's load the data set - this is the EGRA file
!wget --no-check-certificate 'https://docs.google.com/spreadsheets/d/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU/export?format=csv' -O Datareading.csv


--2025-04-09 23:21:22--  https://docs.google.com/spreadsheets/d/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU/export?format=csv
Resolving docs.google.com (docs.google.com)... 172.217.203.139, 172.217.203.138, 172.217.203.102, ...
Connecting to docs.google.com (docs.google.com)|172.217.203.139|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: https://doc-0c-7s-sheets.googleusercontent.com/export/54bogvaave6cua4cdnls17ksc4/cc4ark5iu0ejcsskaqvgdv43qo/1744240880000/113781219181981321798/*/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU?format=csv [following]
--2025-04-09 23:21:23--  https://doc-0c-7s-sheets.googleusercontent.com/export/54bogvaave6cua4cdnls17ksc4/cc4ark5iu0ejcsskaqvgdv43qo/1744240880000/113781219181981321798/*/1Rq8nK25uEKsjwwcu0Sr1IF1oBD6N2BwbmFW5VeiGAfU?format=csv
Resolving doc-0c-7s-sheets.googleusercontent.com (doc-0c-7s-sheets.googleusercontent.com)... 173.194.210.132, 2607:f8b0:400c:c0f::84
Connecting to doc-0c-7s-sheets.googleuser

In [4]:
egra = pd.read_csv('Datareading.csv')#let's take a look at this
egra.describe()
pd.set_option('display.max_columns', None)


<ipython-input-4-afdb31daaa64>:1: DtypeWarning: Columns (15,18,19,20,21,22,23,24,25,27,28,30,31,32,33,34,35,36,37,38,39,44,45,46,50,52,58,59,60,67,69,70,71,80,90,95,96,100,101,102,104,108,109,110,115,121,128,132,133,134,136,137,138,139,140,141,146,147,148,152,155,156,157,158,159,167,168,176,178,179,185,192,200,201,212,219,220,229,230,241,242,243,245,253,254,263,267,268,269,271,276,277,278,280,281,290,291,292,293,294,300,301,305,306,307,311,312,313,314,315,323,324,326,327,328,333,337,338,391,392,393,396,397,452,453,454,455,456,457,458,459,460,461,462,463,464,465,468,469,475,476,478,479,485,486,488,489,500,501,503,504,510,511,513,514,520,521,523,524,533,535,549,551,555,559,563,565,567,569,571,572,573,574,575,576,582,583,584,585,586,587,588,592,593,604,605,608,610,624,626,629,630,631,632,633,634,637,638,639,640,642,643,644,645,647,648,649,650,652,653,654,655,660,661,662,663,664,665,667,670,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,

,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,...,location_schooldistrict_level,location_schoolschool_parent,location_schoolschool_id,location_schoolschool_level,ci_,invent_worditem_at_time,oral_readitem_at_time,sil_readitem_at_time,userprofileitem1female,strata1
count,5275,5275,5275,5275,5275,5275,5275,5275,5275,5275,...,31,31,31,31,1,1,1,1,1214.0,5275
unique,10,54,233,5,5,3,9,233,3,9,...,2,3,4,2,1,1,1,1,3.0,13
top,Жалал-Абадская,Сузакский,20901461,Kyrgyz,Kyrgyz,Rural,K,СШ №66,Treatment,K,...,District,БаткенскаяЛейлекский,1S36560804,School,ci_,invent_word.item_at_time,oral_read.item_at_time,b_oral_read.item_at_time,1.0,2024--Program Kyrgyz
freq,1155,429,41,2874,2874,3797,1772,41,3139,1772,...,30,20,10,30,1,1,1,1,711.0,729


In [5]:
#let's check what we have. I know there's baseline and endline data here. We want just the baseline data, prior to program interventions
egra['g2_year'].value_counts()


,count
g2_year,
2024,2692
2021,1559
2021,1023
Grade 2 Read/Math Year,1


In [6]:
#Strange. I have two identical values for 2021. what's up?
egra['g2_year'].apply(type)
egra['g2_year'].dtypes

#Looks like I have strings and integers...I want to convery this all to string


,g2_year
0,<class 'str'>
1,<class 'str'>
2,<class 'str'>
3,<class 'str'>
4,<class 'str'>
...,...
5270,<class 'int'>
5271,<class 'int'>
5272,<class 'int'>
5273,<class 'int'>


dtype('O')

In [7]:
#let's see if we can convert all our "2021"'s to a string so its the same value
egra['g2_year'] = egra['g2_year'].apply(lambda x: '2021' if str(x) == '2021' else x)


In [8]:
#let's run it again, looks like it worked!
egra['g2_year'].value_counts()


,count
g2_year,
2024,2692
2021,2582
Grade 2 Read/Math Year,1


In [9]:
#now I can finally filter to the data I want.
base = egra[egra['g2_year'] == '2021']

In [10]:
base.count()
#bingo!

,0
region,2582
district,2582
school_code,2582
language,2582
language_name,2582
...,...
invent_worditem_at_time,0
oral_readitem_at_time,0
sil_readitem_at_time,0
userprofileitem1female,0


In [11]:
base.groupby('region').size()

#Breakdown by region, and by group within the baseline study of 2500 observations.
#will need to rename this, but gives some nice quick insights as to what i'll be looking at for my roll up by region!


,0
region,
Баткенская,301
Жалал-Абадская,580
Иссык-Кульская,230
Нарынская,96
Ошская,437
Таласская,140
Чуйская,341
г. Бишкек,318
г. Ош,139


In [12]:
base

,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

##Cleaning Dataset for Regional Demographics

In [13]:
moes = read_pdf('moesreport.pdf', pages='19-20',multiple_tables=True, stream=True)


In [14]:
moes = pd.DataFrame(moes[1])


In [15]:
# Move column names into the first row
moes.loc[-1] = moes.columns  # Shift column names into row 0
moes.index = moes.index + 1  # Shift index to make space
moes = moes.sort_index().reset_index(drop=True)  # Reset index

# Rename the first column if necessary
moes = moes.rename(columns={moes.columns[0]: "Ethnicity"})



In [16]:
moes

,Ethnicity,7 037 590,570 898,1 311 007,538 384,308 348,1 460 425,273 509,1 068 702,1 145 044,361 273,Unnamed: 0
0,Total population,7 037 590,570 898,1 311 007,538 384,308 348,1 460 425,273 509,1 068 702,1 145 044,361 273,Unnamed: 0
1,including:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kyrgyz,5 470 806,451 422,967 355,492 452,307 095,1 003 764,260 038,798 347,975 128,215 205,Batken oblast
3,Russians,277 646,2 021,5 062,27 424,88,1 254,3 257,124 640,110 439,3 461,NaN
4,Uzbeks,995 454,78 314,321 461,3 463,291,421 519,1 006,17 480,15 899,136 021,Jalal-Abad
5,Ukrainians,3 257,10,88,147,-,14,55,1 715,1 199,29,oblast
6,Germans,2 747,2,55,85,6,1,143,1 768,671,16,NaN
7,Tatars,11 353,479,1 027,1 112,78,479,121,3 109,4 324,624,Issyk-Kul oblast
8,Kazakhs,28 389,317,898,5 976,190,823,2 141,10 554,7 176,314,NaN
9,Armenians,494,3,124,10,-,51,1,104,194,7,NaN


In [17]:
# Define new column names
new_column_names = {
    "Ethnicity": "Ethnicity",
    "7 037 590": "Total Population",
    "570 898": "Batken",
    "1 311 007": "Jalal-Abad",
    "538 384": "Issyk-Kul",
    "308 348": "Naryn",
    "1 460 425": "Osh",
    "273 509": "Talas",
    "1 068 702": "Chui",
    "1 145 044": "Bishkek",
    "361 273": "Osh city"
}

# Rename the columns in the DataFrame
moes = moes.rename(columns=new_column_names)

# Drop the "Unnamed: 0" column if it still exists
moes = moes.drop(columns=["Unnamed: 0"], errors="ignore")

# Display the updated DataFrame
moes

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Total population,7 037 590,570 898,1 311 007,538 384,308 348,1 460 425,273 509,1 068 702,1 145 044,361 273
1,including:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kyrgyz,5 470 806,451 422,967 355,492 452,307 095,1 003 764,260 038,798 347,975 128,215 205
3,Russians,277 646,2 021,5 062,27 424,88,1 254,3 257,124 640,110 439,3 461
4,Uzbeks,995 454,78 314,321 461,3 463,291,421 519,1 006,17 480,15 899,136 021
5,Ukrainians,3 257,10,88,147,-,14,55,1 715,1 199,29
6,Germans,2 747,2,55,85,6,1,143,1 768,671,16
7,Tatars,11 353,479,1 027,1 112,78,479,121,3 109,4 324,624
8,Kazakhs,28 389,317,898,5 976,190,823,2 141,10 554,7 176,314
9,Armenians,494,3,124,10,-,51,1,104,194,7


In [18]:
moes.drop([1,10,16,22,28,30], inplace=True)
# Define specific ethnicity name replacements
ethnicity_replacements = {
    "nationalities": "Other nationalities",
    "and pakistan": "Indian and Pakistani",
    # Add more replacements as needed
}

# Apply the replacements
moes["Ethnicity"] = moes["Ethnicity"].replace(ethnicity_replacements)


In [19]:
for col in moes.columns[1:]:  # Skip 'Ethnicity' column
    moes[col] = moes[col].astype(str).str.replace(" ", "").str.replace("-", "0") #replace '-' with 0 for missing values. you may want to use NaN instead
    moes[col] = pd.to_numeric(moes[col], errors='coerce').fillna(0).astype(int) # Convert to numeric, handle errors, fill NaN with 0, then to int


In [20]:
moes['Ethnicity'] = moes['Ethnicity'].str.replace('Total', 'Sum') #this will do it for any ethnicity with 'Total' in the name
moes

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,7037590,570898,1311007,538384,308348,1460425,273509,1068702,1145044,361273
2,Kyrgyz,5470806,451422,967355,492452,307095,1003764,260038,798347,975128,215205
3,Russians,277646,2021,5062,27424,88,1254,3257,124640,110439,3461
4,Uzbeks,995454,78314,321461,3463,291,421519,1006,17480,15899,136021
5,Ukrainians,3257,10,88,147,0,14,55,1715,1199,29
6,Germans,2747,2,55,85,6,1,143,1768,671,16
7,Tatars,11353,479,1027,1112,78,479,121,3109,4324,624
8,Kazakhs,28389,317,898,5976,190,823,2141,10554,7176,314
9,Armenians,494,3,124,10,0,51,1,104,194,7
11,Tajiks,60752,36921,7153,213,23,8626,53,5208,1743,812


In [21]:
regselect = ['Sum population','Kyrgyz','Russians','Uzbeks','Tajiks']
regethnics = moes[moes['Ethnicity'].isin(regselect)]

In [22]:
regethnics
#table to provide total population, filtered for the ethnicities of interest

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,7037590,570898,1311007,538384,308348,1460425,273509,1068702,1145044,361273
2,Kyrgyz,5470806,451422,967355,492452,307095,1003764,260038,798347,975128,215205
3,Russians,277646,2021,5062,27424,88,1254,3257,124640,110439,3461
4,Uzbeks,995454,78314,321461,3463,291,421519,1006,17480,15899,136021
11,Tajiks,60752,36921,7153,213,23,8626,53,5208,1743,812


In [23]:
regethnics = regethnics.applymap(lambda x: f"{x:,}" if isinstance(x, (int, float)) else x)
regethnics
#We are interested in the four target languages and groups by region, as well as total population


<ipython-input-23-5a03fe27d567>:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  regethnics = regethnics.applymap(lambda x: f"{x:,}" if isinstance(x, (int, float)) else x)


,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,"7,037,590","570,898","1,311,007","538,384","308,348","1,460,425","273,509","1,068,702","1,145,044","361,273"
2,Kyrgyz,"5,470,806","451,422","967,355","492,452","307,095","1,003,764","260,038","798,347","975,128","215,205"
3,Russians,"277,646","2,021","5,062","27,424",88,"1,254","3,257","124,640","110,439","3,461"
4,Uzbeks,"995,454","78,314","321,461","3,463",291,"421,519","1,006","17,480","15,899","136,021"
11,Tajiks,"60,752","36,921","7,153",213,23,"8,626",53,"5,208","1,743",812


Remaining is to download the report, remove the indexes, and add a title "Regional population breakout by region and demographics". This will then be transferred over to the appendix.

In [24]:
regethnics.to_csv('moesreport.csv')
files.download('moesreport.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Cleaning EGRA Dataset

In [25]:
base #Cleaning dataset
pd.set_option('display.max_columns', None)


,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

In [26]:
#replace/map/recode
base['region'].replace({
    'Ошская': 'Osh',
    'г. Ош' : 'Osh_city',
    'Чуйская':'Chui',
    'Таласская':'Talas',
    'Иссык-Кульская':'Issyk-Kul',
    'г. Бишкек':'Bishkek',
    'Жалал-Абадская': 'Jalal-Abad',
    'Нарынская':'Naryn',
    'Баткенская':'Batken'}, inplace=True)
base.head(3)# names fixed,


<ipython-input-26-73aea0ebcfff>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  base['region'].replace({
<ipython-input-26-73aea0ebcfff>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['region'].replace({


,region,district,school_code,language,language_name,urbanrural,loi,school_name,treatment,loi_name,loi_kyrgyz,loi_russian,loi_uzbek,loi_tajik,stage1,fpc1,in_G2_master,in_G4_master,n_sampled,n_g4_sampled,pop_strata1,n_g2f_sampled,n_g2m_sampled,Dup_id,g_tot,n_classes,strata,n_g4f_sampled,n_g4m_sampled,grade,n_g2_sampled,pop_g2_strata1,pop_g4_strata1,g2g4_tot,pop_g2g4_strata1,g2_tot,g4_tot,g2_classes,g4_classes,in_ht,ht_start_time,ht_end_time,ht_time_start,ht_enumerator,ht_gps_latitude,ht_gps_longitude,ht_gps_accuracy,userprofileitem1first_name,userprofileitem1last_name,sh_consent,sh_female,sh_position,sh_position_other,sh_1,sh_2_1,sh_2_2,sh_2_3,sh_2_4,sh_2_777,sh_2_888,sh_2_other,sh_3_1,sh_3_2,sh_3_3,sh_3_4,sh_3_5,sh_3_6,sh_3_8,sh_3_7,sh_3_777,sh_3_888,sh_3_other,sh_4_1,sh_4_2,sh_4_3,sh_4_4,sh_4_5,sh_4_6,sh_4_7,sh_4_777,sh_4_888,sh_4_other,sh_5,sh_6,sh_7,sh_8_1,sh_8_2,sh_8_3,sh_8_4,sh_8_5,sh_8_888,sh_9,sh_10,date_end,ht_time_end,sh_8_6,ht_year,ht_date,ht_date_char,ht_dayofweek,ht_month,ht_day,sh_minutes_assess,_merge_Master_HT,in_l,l_start_time,l_end_time,l_enumerator,l_gps_latitude,l_gps_longitude,l_gps_accuracy,l_consent,l_female,l1,l2,l3,l4,l5_1,l5_2,l5_3,l5_4,l5_888,l6,l7,l8,l9,l_time_end,l_time_start,l_year,l_date,l_date_char,l_dayofweek,l_month,l_day,l_minutes_assess,_merge_Master_Lib,n_schools_sampled,wt1,sum_wt1,scale_wt1,wt_stage1,in_t,t_start_time,t_end_time,t_time_start,t_enumerator,t_gps_latitude,t_gps_longitude,t_gps_accuracy,t_consent,t_female,tq_language,tq_language_other,tq_1,tq_2,tq_3,tq_4_1,tq_4_2,tq_4_3,tq_4_888,tq_5,tq_6_1,tq_6_2,tq_6_3,tq_6_4,tq_6_5,tq_6_6,tq_6_000,tq_6_888,tq_7,tq_8,tq_9,tq_10_1,tq_10_2,tq_10_3,tq_10_4,tq_10_5,tq_10_6,tq_10_000,tq_10_888,tq_11_1,tq_11_2,tq_11_3,tq_11_4,tq_11_777,tq_11_888,tq_11_other,tq_12_1,tq_12_2,tq_12_3,tq_12_4,tq_12_5,tq_12_888,tq_13_1,tq_13_2,tq_13_3,tq_13_4,tq_13_5,tq_13_6,tq_13_7,tq_13_000,tq_13_888,tq_14,tq_15,tq_16,tq_17_1,tq_17_2,tq_17_3,tq_17_4,tq_17_5,tq_17_6,tq_17_777,tq_17_888,tq_17_other,tq_18_1,tq_18_2,tq_18_3,tq_18_4,tq_18_777,tq_18_000,tq_18_888,tq_18_other,tq_19,tq_20_1,tq_20_2,tq_20_3,tq_20_4,tq_20_5,tq_20_6,tq_20_777,tq_20_888,tq_21,tq_21_other,tq_22,tq_23_1,tq_23_2,tq_23_3,tq_23_4,tq_23_5,tq_23_6,tq_23_7,tq_23_777,tq_23_888,tq_23_other,tq_24,tq_24_other,tq_25_1,tq_25_2,tq_25_3,tq_25_4,tq_25_5,tq_25_6,tq_25_7,tq_25_777,tq_25_888,tq_26,tq_27,tq_28,tq_29,tq_30,tq_31,tq_32,t_time_end,t_year,t_date,t_date_char,t_dayofweek,t_month,t_day,t_minutes_assess,_merge_Master_Teach,in_ci,ci_start_time,ci_end_time,ci_time_start,ci_enumerator,ci_gps_latitude,ci_gps_longitude,ci_gps_accuracy,ci_language,males_enrolled,females_enrolled,ci_1_1,ci_1_2,ci_1_3,ci_1_4,ci_1_5,ci_1_6,ci_1_7,ci_1_8,ci_1_000,ci_2,ci_3,ci_4,ci_5,ci_6,ci_7,ci_8,ci_9,ci_time_end,userprofileid,ci_year,ci_date,ci_date_char,ci_dayofweek,ci_month,ci_day,ci_minutes_assess,_merge_Master_ClassInv,strata2,stage2,fpc2,wt2,wt12,wt_stage2,in_g2,sil_read_comp_score_pcnt80,sil_read_comp_attempted_pcnt80,sil_read_comp1,sil_read_comp2,sil_read_comp3,sil_read_comp4,sil_read_comp5,sil_read_comp_score,sil_read_comp_score_pcnt,sil_read_comp_score_zero,sil_read_comp_attempted,sil_read_comp_attempted_pcnt,g2_year,g2_month,g2_date,id,female,age,g2_start_time,g2_end_time,g2_consent,cnonwpm,orf,read_comp_score_pcnt80,read_comp_attempted_pcnt80,invent_word1,invent_word2,invent_word3,invent_word4,invent_word5,invent_word6,invent_word7,invent_word8,invent_word9,invent_word10,invent_word11,invent_word12,invent_word13,invent_word14,invent_word15,invent_word16,invent_word17,invent_word18,invent_word19,invent_word20,invent_word21,invent_word22,invent_word23,invent_word24,invent_word25,invent_word26,invent_word27,invent_word28,invent_word29,invent_word30,invent_word31,invent_word32,invent_word33,invent_word34,invent_word35,invent_word36,invent_word37,invent_word38,invent_word39,invent_word40,invent_word41,invent_word42,invent_word43,invent_word44,invent_word45,invent_word46,invent_word47,invent_word48,invent_word49,invent_word50,i

In [27]:
#Let's get this ordered alphabetically
base.sort_values(by='region',ascending=True,inplace=True)

<ipython-input-27-ce4a05c3bb33>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base.sort_values(by='region',ascending=True,inplace=True)


In [28]:
base['oral_read_score_pcnt'] = base['oral_read_score_pcnt'].fillna(0).round().astype(int)
base['oral_read_score_pcnt'].value_counts('')


<ipython-input-28-0d27f7ab883d>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['oral_read_score_pcnt'] = base['oral_read_score_pcnt'].fillna(0).round().astype(int)


,count
oral_read_score_pcnt,
100,424
98,226
96,120
70,58
94,50
...,...
69,3
31,3
56,2


oral_read_score_pcnt	- DONE
g2_math_overall_score_pcnt - DONE

Now let's do some coding to clean up these remaining items that either could have NaN or odd numbers. let's go one by one.

oral_read_attempted_pcnt
read_comp_score_pcnt
word_score
word_score_pcnt
invent_word_score_pcnt


In [29]:
base['oral_read_attempted_pcnt'].value_counts('')
#same issue

,count
oral_read_attempted_pcnt,
100.0,571
100,399
98,205
98,134
96,134
...,...
66,1
63,1
47,1


In [30]:
base['oral_read_attempted_pcnt'] = base['oral_read_attempted_pcnt'].fillna(0).round().astype(int)
base['oral_read_attempted_pcnt'].value_counts()

<ipython-input-30-d097bc5717a9>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['oral_read_attempted_pcnt'] = base['oral_read_attempted_pcnt'].fillna(0).round().astype(int)


,count
oral_read_attempted_pcnt,
100,970
98,339
96,202
97,116
95,115
...,...
29,1
45,1
49,1


In [31]:
base['read_comp_score_pcnt'].value_counts()


,count
read_comp_score_pcnt,
100.0,290
20,265
80.0,265
40.0,262
60,251
0.0,225
80,197
20,179
100,174


In [32]:
base['read_comp_score_pcnt'] = base['read_comp_score_pcnt'].fillna(0).round().astype(int)
base['read_comp_score_pcnt'].value_counts()
#looks cleaner to me!
base['list_comp_score_pcnt'] = base['list_comp_score_pcnt'].fillna(0).round().astype(int)
base['list_comp_score_pcnt'].value_counts()


<ipython-input-32-4d165f29d670>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['read_comp_score_pcnt'] = base['read_comp_score_pcnt'].fillna(0).round().astype(int)


,count
read_comp_score_pcnt,
100,464
80,462
20,444
60,419
40,405
0,388


<ipython-input-32-4d165f29d670>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['list_comp_score_pcnt'] = base['list_comp_score_pcnt'].fillna(0).round().astype(int)


,count
list_comp_score_pcnt,
100,661
80,589
60,517
40,343
0,257
20,215


In [33]:
base['word_score'].value_counts()
#what is going on here.
base['word_score'].dtype

,count
word_score,
4,308
3,288
6,274
5,272
2,206
3,196
2,186
1,173
4,172


dtype('O')

In [34]:
base['word_score'] = base['word_score'].fillna(0).round().astype(int)
base['word_score'].value_counts() #better!

<ipython-input-34-b759158afdb2>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['word_score'] = base['word_score'].fillna(0).round().astype(int)


,count
word_score,
3,484
4,480
5,443
6,401
2,392
1,315
0,67


In [35]:
base['word_score_pcnt'].value_counts()

,count
word_score_pcnt,
67,308
50,288
100,274
83,272
33,206
50,196
33,186
17,173
67,172


In [36]:
base['word_score_pcnt'] = base['word_score_pcnt'].fillna(0).round().astype(int)
base['word_score_pcnt'].value_counts() #much cleaner!

<ipython-input-36-6d9ff4f1bd4b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['word_score_pcnt'] = base['word_score_pcnt'].fillna(0).round().astype(int)


,count
word_score_pcnt,
50,484
67,480
83,443
100,401
33,392
17,315
0,67


In [37]:
base['invent_word_score_pcnt'].value_counts()

,count
invent_word_score_pcnt,
46.0,52
62,50
94,46
48.0,46
54,45
...,...
10,8
2,7
4,7


In [38]:
base['invent_word_score_pcnt'] = base['invent_word_score_pcnt'].fillna(0).round().astype(int)
base['invent_word_score_pcnt'].value_counts() #fixed!

<ipython-input-38-0719fad180d7>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['invent_word_score_pcnt'] = base['invent_word_score_pcnt'].fillna(0).round().astype(int)


,count
invent_word_score_pcnt,
46,78
64,76
56,75
60,75
62,73
54,72
98,70
52,69
44,68


In [39]:
base['invent_word_score'] = base['invent_word_score'].fillna(0).round().astype(int)
base['word_score'] = base['word_score'].fillna(0).round().astype(int)
base['oral_read_score'] = base['oral_read_score'].fillna(0).round().astype(int)




<ipython-input-39-8c860fde7447>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['invent_word_score'] = base['invent_word_score'].fillna(0).round().astype(int)
<ipython-input-39-8c860fde7447>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base['word_score'] = base['word_score'].fillna(0).round().astype(int)
<ipython-input-39-8c860fde7447>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [40]:
 #subset
 base = base[[
    'region',
    'district',
    'language',
    'urbanrural',
    'female',
    'age',
    'g2_year',
    'oral_read_score',
    'oral_read_score_pcnt',
    'read_comp_score_pcnt',
    'word_score',
    'word_score_pcnt',
    'invent_word_score',
    'invent_word_score_pcnt',
    'list_comp_score_pcnt',
    ]]
base

,region,district,language,urbanrural,female,age,g2_year,oral_read_score,oral_read_score_pcnt,read_comp_score_pcnt,word_score,word_score_pcnt,invent_word_score,invent_word_score_pcnt,list_comp_score_pcnt
385,Batken,Кадамжайский,Kyrgyz,Rural,Female,8,2021,55,98,100,2,33,48,96,40
1136,Batken,Баткенский,Kyrgyz,Rural,Female,9,2021,26,46,40,2,33,24,48,60
1138,Batken,Баткенский,Kyrgyz,Rural,Female,9,2021,56,100,100,3,50,23,46,80
272,Batken,Кадамжайский,Kyrgyz,Rural,Female,8,2021,56,100,100,2,33,48,96,100
1140,Batken,Баткенский,Kyrgyz,Rural,Female,8,2021,29,52,40,4,67,20,40,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
794,Talas,Бакай-Атинский,Russian,Rural,Female,9,2021,23,43,20,1,17,22,44,0
1642,Talas,Кара-Бууринский,Kyrgyz,Rural,Male,8,2021,21,38,20,6,100,18,36,60
799,Talas,Бакай-Атинский,Russian,Rural,Male,7,2021,37,70,20,5,83,35,70,40
778,Talas,Кара-Бууринский,Russian,Rural,Female,7,2021,45,85,80,4,67,26,52,80


# EGRA Initial Tables and Graphs

##Sample Distribution by region and language
1. By Region, we have a relatively healthy sample size that is repreresentative enough to compare between regions. All good here!
2. By language, a few items jump out as concerns.
  **Osh City** having only 10 uzbek test takers. This makes it unlikely (this is one school) that we can make any statements on the Uzbek/Osh City grouping.  Were uzbek grade 2 students making the choice to do Kyrgyz or Russian instead of their native language - and did that bring scores down in this region?
  **Osh** only having 40 test takers in Uzbek. This is enough, but at the same time this isn't seemingly representative of the demographic breakdown. Did individuals who speak Uzbek at home take tests in Kyrgyz and Russian?
  **Naryn** having only 16 test takers in Russian means it isn't likely we can review this language/region group

3. Noting, in the EGRA 2021 Baselinereport the team indicates that all Uzbek language schools were visited in country, and this would be a first time assessment of Uzbek schools. The authors warn about drawing larger representative conclusions on Uzbek performance compared to Kyrgyz and Russian due to the change in sample size. Batken however has a sizeable and appropriate sample size, as does Jalal-Abad, and to a much lesser extent, Osh. The literature further indicates that in the south, as a result of the reduction of status of Uzbek language in the country (removal from NST testing options, reduced uzbek language options, focus on krygyz as official language over Russian, while Russian being a language of economic opportunity) - more uzbek students speak uzbek at home, but take classes in other languages.

In [41]:
baselinecount = base.groupby(by=['region','language']).size().rename('count')
baselinecount



region      language
Batken      Kyrgyz      140
            Russian      38
            Tajik        30
            Uzbek        93
Bishkek     Kyrgyz      129
            Russian     189
Chui        Kyrgyz      197
            Russian     144
Issyk-Kul   Kyrgyz      166
            Russian      64
Jalal-Abad  Kyrgyz      311
            Russian     159
            Uzbek       110
Naryn       Kyrgyz       80
            Russian      16
Osh         Kyrgyz      283
            Russian     114
            Uzbek        40
Osh_city    Kyrgyz       19
            Russian     110
            Uzbek        10
Talas       Kyrgyz      100
            Russian      40
Name: count, dtype: int64

In [42]:
title = "Sample Distribution by Region & Language"

# Saving the DataFrame to a CSV file
filename = 'baselinecount_with_title.csv'
with open(filename, 'w') as f:
    f.write(title + '\n')  # Add the title first
    baselinecount.to_csv(f, index=True)

41

In [43]:
files.download('baselinecount_with_title.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Average Score by Region & Language, Pivot Table
A bunch of interesting items pop out here.
1. Oral Reading Subtask + Benchmark Percentages
2. Reading Comprehension + Benchmark Percentages
3. Silent Reading:
4. Invented Word Score:
5. Listening Score:


In [86]:
oral_read_score_gt_30 = base[base['oral_read_score'] > 30].groupby(['region', 'language']).size() / base.groupby(['region', 'language']).size()

read_comp_score_80 = base[base['read_comp_score_pcnt'] >= 80].groupby(['region', 'language']).size() / base.groupby(['region', 'language']).size()

# Create the pivot table
baselinemean = base.pivot_table(
    index=['region', 'language'],
    values=[
        'oral_read_score_pcnt',
        'read_comp_score_pcnt',
        'word_score_pcnt',

    ],
    aggfunc='mean'
)

# Add the proportions as new columns to the pivot table
baselinemean['oral_read_score_gt_30'] = oral_read_score_gt_30
baselinemean['read_comp_score_80'] = read_comp_score_80

# Round the results
baselinemean = baselinemean.round(2)
baselinemean

oral_read_score_pcnt  read_comp_score_pcnt  \
region     language                                               
Batken     Kyrgyz                   78.71                 64.14   
           Russian                  54.21                 27.37   
           Tajik                    65.70                 41.33   
           Uzbek                    83.20                 82.80   
Bishkek    Kyrgyz                   78.66                 63.41   
           Russian                  90.30                 79.89   
Chui       Kyrgyz                   71.76                 58.88   
           Russian                  72.76                 46.11   
Issyk-Kul  Kyrgyz                   76.13                 59.64   
           Russian                  70.05                 58.75   
Jalal-Abad Kyrgyz                   58.68                 45.72   
           Russian                  61.77                 34.72   
           Uzbek                    46.75                 40.00   
Naryn      Kyrgyz                   70.55                 56.50   
           Russian                  83.12                 60.00   
Osh        Kyrgyz                   54.58                 37.95   
           Russian                  48.84                 18.95   
           Uzbek                    70.62                 66.00   
Osh_city   Kyrgyz                   69.84                 61.05   
           Russian                  71.86                 46.36   
           Uzbek                    45.60                 46.00   
Talas      Kyrgyz                   74.49                 56.00   
           Russian                  73.95                 49.00   

                     word_score_pcnt  oral_read_score_gt_30  \
region     language                                           
Batken     Kyrgyz              54.16                   0.77   
           Russian             37.34                   0.37   
           Tajik               72.23                   0.70   
           Uzbek               87.72                   0.83   
Bishkek    Kyrgyz              57.89                   0.80   
           Russian             65.68                   0.95   
Chui       Kyrgyz              78.37                   0.71   
           Russian             48.86                   0.74   
Issyk-Kul  Kyrgyz              54.92                   0.77   
           Russian             56.52                   0.72   
Jalal-Abad Kyrgyz              60.85                   0.56   
           Russian             47.72                   0.55   
           Uzbek               61.69                   0.43   
Naryn      Kyrgyz              62.06                   0.68   
           Russian             70.81                   0.94   
Osh        Kyrgyz              48.19                   0.50   
           Russian             33.39                   0.38   
           Uzbek               88.30                   0.70   
Osh_city   Kyrgyz              78.05                   0.63   
           Russian             55.32                   0.75   
           Uzbek               55.10                   0.30   
Talas      Kyrgyz              64.80                   0.76   
           Russian             53.30                   0.72   

                     read_comp_score_80  
region     language                      
Batken     Kyrgyz                  0.51  
           Russian                 0.05  
           Tajik                   0.23  
           Uzbek                   0.71  
Bishkek    Kyrgyz                  0.52  
           Russian                 0.74  
Chui       Kyrgyz                  0.46  
           Russian                 0.20  
Issyk-Kul  Kyrgyz                  0.46  
           Russian                 0.34  
Jalal-Abad Kyrgyz                  0.32  
           Russian                 0.16  
           Uzbek                   0.14  
Naryn      Kyrgyz                  0.39  
           Russian                 0.25  
Osh        Kyrgyz                  0.25  
           Ru

In [87]:
# prompt: organize baseline mean columns as such: Oral_read_score_pcnt, oral_read_score_gt_40, read_comp_score_pcnt, read_comp_score_80, word_score_pcnt

# Reorder columns in baselinemean DataFrame
baselinemean = baselinemean[[
    'oral_read_score_pcnt',
    'oral_read_score_gt_30',
    'read_comp_score_pcnt',
    'read_comp_score_80',
    'word_score_pcnt'
]]

baselinemean


oral_read_score_pcnt  oral_read_score_gt_30  \
region     language                                                
Batken     Kyrgyz                   78.71                   0.77   
           Russian                  54.21                   0.37   
           Tajik                    65.70                   0.70   
           Uzbek                    83.20                   0.83   
Bishkek    Kyrgyz                   78.66                   0.80   
           Russian                  90.30                   0.95   
Chui       Kyrgyz                   71.76                   0.71   
           Russian                  72.76                   0.74   
Issyk-Kul  Kyrgyz                   76.13                   0.77   
           Russian                  70.05                   0.72   
Jalal-Abad Kyrgyz                   58.68                   0.56   
           Russian                  61.77                   0.55   
           Uzbek                    46.75                   0.43   
Naryn      Kyrgyz                   70.55                   0.68   
           Russian                  83.12                   0.94   
Osh        Kyrgyz                   54.58                   0.50   
           Russian                  48.84                   0.38   
           Uzbek                    70.62                   0.70   
Osh_city   Kyrgyz                   69.84                   0.63   
           Russian                  71.86                   0.75   
           Uzbek                    45.60                   0.30   
Talas      Kyrgyz                   74.49                   0.76   
           Russian                  73.95                   0.72   

                     read_comp_score_pcnt  read_comp_score_80  word_score_pcnt  
region     language                                                             
Batken     Kyrgyz                   64.14                0.51            54.16  
           Russian                  27.37                0.05            37.34  
           Tajik                    41.33                0.23            72.23  
           Uzbek                    82.80                0.71            87.72  
Bishkek    Kyrgyz                   63.41                0.52            57.89  
           Russian                  79.89                0.74            65.68  
Chui       Kyrgyz                   58.88                0.46            78.37  
           Russian                  46.11                0.20            48.86  
Issyk-Kul  Kyrgyz                   59.64                0.46            54.92  
           Russian                  58.75                0.34            56.52  
Jalal-Abad Kyrgyz                   45.72                0.32            60.85  
           Russian                  34.72                0.16            47.72  
           Uzbek                    40.00                0.14            61.69  
Naryn      Kyrgyz                   56.50                0.39            62.06  
           Russian                  60.00                0.25            70.81  
Osh        Kyrgyz                   37.95                0.25            48.19  
           Russian                  18.95                0.05            33.39  
           Uzbek                    66.00                0.42            88.30  
Osh_city   Kyrgyz                   61.05                0.53            78.05  
           Russian                  46.36                0.22            55.32  
           Uzbek                    46.00                0.10            55.10  
Talas      Kyrgyz                   56.00                0.41            64.80  
           Russian                  49.00                0.25            53.30

##Violin Charts - Distribution by Subtask

In [111]:
# Define the subtask (column name)
subtask = "oral_read_score_pcnt"

# Filter dataframe for only this subtask
oralread = base[["region", subtask]].dropna()  # Remove missing values

# Create violin plot with embedded boxplots
fig = px.violin(
    oralread,
    x="region",
    y=subtask,
    color="region",
    box=True,               # show internal boxplot
    points="all",           # show all datapoints
)
# 🧠 Here's the working hack to add black outlines to the boxplots
for trace in fig.data:
    if trace.type == 'violin':
        trace.box = dict(line=dict(color='black'))

fig.update_layout(
    yaxis_title="Score",
    paper_bgcolor="white",
    font=dict(color="black"),  # Global font color for all text elements
    legend=dict(
        font=dict(size=20),         # Increase legend item font size
    ),
    xaxis=dict(
        tickfont=dict(size=20),     # Increase y-axis tick label font size
    )
)

In [112]:
# Define the subtask (column name)
subtask2 = "read_comp_score_pcnt"

# Filter dataframe for only this subtask
readcomp = base[["region", subtask2]].dropna()

# Create the violin plot
fig = px.violin(readcomp, x="region", y=subtask2, color="region",
                box=True,
                points="all",
                title=f"Score Distribution for {subtask2} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)
# 🧠 Here's the working hack to add black outlines to the boxplots
for trace in fig.data:
    if trace.type == 'violin':
        trace.box = dict(line=dict(color='black'))

fig.update_layout(
    yaxis_title="Score",
    paper_bgcolor="white",
    font=dict(color="black"),  # Global font color for all text elements
    legend=dict(
        font=dict(size=20),         # Increase legend item font size
    ),
    xaxis=dict(
        tickfont=dict(size=20),     # Increase y-axis tick label font size
    )
)

In [113]:
# Define the subtask (column name)
subtask3 = "word_score_pcnt"

# Filter dataframe for only this subtask
wordscore = base[["region", subtask3]].dropna()  # Remove missing values

# Create the violin plot
fig = px.violin(wordscore, x="region", y=subtask3, color="region",
                box=True,  # Adds a mini box plot inside
                points="all",  # Show all data points
                title=f"Score Distribution for {subtask3} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)
# 🧠 Here's the working hack to add black outlines to the boxplots
for trace in fig.data:
    if trace.type == 'violin':
        trace.box = dict(line=dict(color='black'))

fig.update_layout(
    yaxis_title="Score",
    paper_bgcolor="white",
    font=dict(color="black"),  # Global font color for all text elements
    legend=dict(
        font=dict(size=20),         # Increase legend item font size
    ),
    xaxis=dict(
        tickfont=dict(size=20),     # Increase y-axis tick label font size
    )
)

In [48]:
# Define the subtask (column name)
subtask4 = "invent_word_score_pcnt"

# Filter dataframe for only this subtask
inventwordscore = base[["region", subtask4]].dropna()  # Remove missing values

# Create the violin plot
fig = px.violin(inventwordscore, x="region", y=subtask4, color="region",
                box=True,  # Adds a mini box plot inside
                points="all",  # Show all data points
                title=f"Score Distribution for {subtask4} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)



In [49]:
# Define the subtask (column name)
subtask5 = 'list_comp_score_pcnt'


# Filter dataframe for only this subtask
wordscore = base[["region", subtask5]].dropna()  # Remove missing values

# Create the violin plot
fig = px.violin(wordscore, x="region", y=subtask5, color="region",
                box=True,  # Adds a mini box plot inside
                points="all",  # Show all data points
                title=f"Score Distribution for {subtask5} Across Regions")

# Customize appearance
fig.update_layout(
    xaxis_title="Region",
    yaxis_title="Score",
    legend_title="Region",
    paper_bgcolor="white",
    font=dict(color="black")
)

**Oral Reading Score**
We can see that Bishkek has one heck of a tail at the upper end, almost looks like a glassblowers pipe! And the box plot is super tight!

Other regions have similar candles, with the distribution towards the top.
The interesting outliers here appear to be Jalal-Abad, Osh, and Osh City. These three , in reviewing the box plot, have a much wider range and we can see don't have the "tail" appearing at the end of the violin chart. Definitely something up with these three regions as a whole!

**Invented Word**
Aside from Bishkek, we're seeing a pretty standard distribution across the board.

**Reading Comp Score Review**
Reading comprehension followed the story/oral reading score.
If students did not fail to read the first line of words, they would be asked questions. If they did not reach certain parts of the story in time, they might not be asked certain questions.
We would expect, as a result, the results might be a bit smoother as some students wouldn't get as far along to receive the next reading comprehension question.

Osh, Talas, and Jalal Abad show a strong concentration and grouping at the bottom of the violin chart. Osh especially is showing some interesting, and perhaps troubling, results. The other regions appear to have a more normal distribution in comparison.

**Word Score Pcnt**
This subtask involved silent reading comprehension.
We're seeing similar patterns to the prior two. However, a really, really interesting note is that the students in Chui seemed to have done a bit better, with a wider distribution than all the other regions, even Bishkek, which did an amazing job on the oral reading and oral reading comprehension questions compared to the other regions. Interesting!


#Statistics

##Two Way ANOVA - between region and language

To summarize the tests run below; a two way ANOVA run on each subtask shows that 1. Region is significant 2. Language is significant. and 3, and most importantly: the interaction between region and language is significant, and differs by region. This is true for all subtasks.

This means I need to follow up digging into each region to see where language is, and isn't, significant.

###Oral Reading Score Two Way ANOVA

In [115]:
# Fit the two-way ANOVA model
model = ols('oral_read_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Convert the ANOVA table into a DataFrame (if it's not already)
anovaoralreadtwo = pd.DataFrame(anova_table)
anovaoralreadtwo

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 8, but rank is 3

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 24, but rank is 11



,sum_sq,df,F,PR(>F)
C(region),1.521914e+05,8.0,25.881604,1.723810e-16
C(language),3.910900e+04,3.0,17.735627,2.240837e-08
C(region):C(language),1.617795e+05,24.0,9.170720,2.674677e-16
Residual,1.880959e+06,2559.0,NaN,NaN


###Reading Comprehension Two Way ANOVA

In [116]:
# Fit the two-way ANOVA model
model = ols('read_comp_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Convert the ANOVA table into a DataFrame (if it's not already)
anovareadcomptwo = pd.DataFrame(anova_table)
anovareadcomptwo

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 8, but rank is 3

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 24, but rank is 11



,sum_sq,df,F,PR(>F)
C(region),2.573905e+05,8.0,34.018029,1.485599e-21
C(language),1.137836e+05,3.0,40.101912,7.100919e-18
C(region):C(language),3.066201e+05,24.0,13.508152,1.808191e-25
Residual,2.420270e+06,2559.0,NaN,NaN


###Silent Reading Comprehension Two Way ANOVA

In [117]:
# Fit the two-way ANOVA model
model = ols('word_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

silentreadcomptwo = pd.DataFrame(anova_table)
silentreadcomptwo

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 8, but rank is 3

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 24, but rank is 11



,sum_sq,df,F,PR(>F)
C(region),1.242209e+05,8.0,23.806973,3.408629e-15
C(language),1.781513e+05,3.0,91.047306,6.340733e-39
C(region):C(language),3.112415e+05,24.0,19.883171,5.565458e-39
Residual,1.669056e+06,2559.0,NaN,NaN


###Invented Word Score Two Way ANOVA

In [53]:
# Fit the two-way ANOVA model
model = ols('invent_word_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

                             sum_sq      df          F        PR(>F)
C(region)              7.180573e+04     8.0  15.407385  6.198169e-10
C(language)            1.381784e+04     3.0   7.906397  3.774511e-04
C(region):C(language)  6.565134e+04    24.0   4.695611  3.761868e-07
Residual               1.490769e+06  2559.0        NaN           NaN


/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 8, but rank is 3

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 24, but rank is 11



###List Comp Score Two Way ANOVA

In [54]:
# Fit the two-way ANOVA model
model = ols('list_comp_score_pcnt ~ C(region) + C(language) + C(region):C(language)', data=base).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

# Print the ANOVA table
print(anova_table)

                             sum_sq      df           F        PR(>F)
C(region)              4.338827e+04     8.0    7.343190  6.714720e-05
C(language)            4.100092e+05     3.0  185.043889  8.746655e-76
C(region):C(language)  3.580294e+05    24.0   20.198068  1.204970e-39
Residual               1.890026e+06  2559.0         NaN           NaN


/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 8, but rank is 3

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning:

covariance of constraints does not have full rank. The number of constraints is 24, but rank is 11



##Tukey Test - Significance Between Regions (not reviewing language)

###Oral Reading Score - Regions - Tukey Test

In [55]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['oral_read_score_pcnt'], base['region'])

# Print results
print(tukey_results)
oralregtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
oralregtukey

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
  group1     group2   meandiff p-adj   lower    upper   reject
--------------------------------------------------------------
    Batken    Bishkek   9.8677 0.0003   2.9797  16.7556   True
    Batken       Chui  -3.5262 0.7963 -10.3002   3.2478  False
    Batken  Issyk-Kul  -1.2718 0.9999  -8.7732   6.2295  False
    Batken Jalal-Abad -18.4454    0.0   -24.53 -12.3609   True
    Batken      Naryn  -3.0651 0.9901 -13.1047   6.9745  False
    Batken        Osh -21.1572    0.0 -27.5729 -14.7415   True
    Batken   Osh_city  -6.0131 0.4561 -14.7968   2.7705  False
    Batken      Talas  -1.3752 0.9999 -10.1374   7.3869  False
   Bishkek       Chui -13.3939    0.0  -20.071  -6.7167   True
   Bishkek  Issyk-Kul -11.1395 0.0001 -18.5535  -3.7255   True
   Bishkek Jalal-Abad -28.3131    0.0 -34.2897 -22.3365   True
   Bishkek      Naryn -12.9328 0.0019 -22.9073  -2.9583   True
   Bishkek        Osh -31.0248    0.0 -37.3382 -24.7115

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken,Bishkek,9.8677,0.0003,2.9797,16.7556,True
1,Batken,Chui,-3.5262,0.7963,-10.3002,3.2478,False
2,Batken,Issyk-Kul,-1.2718,0.9999,-8.7732,6.2295,False
3,Batken,Jalal-Abad,-18.4454,0.0000,-24.5300,-12.3609,True
4,Batken,Naryn,-3.0651,0.9901,-13.1047,6.9745,False
5,Batken,Osh,-21.1572,0.0000,-27.5729,-14.7415,True
6,Batken,Osh_city,-6.0131,0.4561,-14.7968,2.7705,False
7,Batken,Talas,-1.3752,0.9999,-10.1374,7.3869,False
8,Bishkek,Chui,-13.3939,0.0000,-20.0710,-6.7167,True
9,Bishkek,Issyk-Kul,-11.1395,0.0001,-18.5535,-3.7255,True


####Oral Reading Score Between Regions, Significance
1. Batken outperformed Osh and Jalal-Abad (Significant as they are the other areas in the south)

2. Bishkek outperformed Chui, issyk kul, jalal abad,naryn, osh, osh-city, talas, and Batken on average. Bishkek a clear leader in oral reading test scores and did significantly better than all other regions, regardless of language.

3. Osh City did not outperform any other region in a significant way.

###Reading Comp Score - Regions - Tukey Test

In [56]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['read_comp_score_pcnt'], base['region'])

# Print results
print(tukey_results)
readregtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
readregtukey

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
  group1     group2   meandiff p-adj   lower    upper   reject
--------------------------------------------------------------
    Batken    Bishkek  10.2175 0.0024   2.2178  18.2172   True
    Batken       Chui  -9.5003 0.0057 -17.3677  -1.6329   True
    Batken  Issyk-Kul  -3.5987 0.9366 -12.3109   5.1134  False
    Batken Jalal-Abad -21.3693    0.0  -28.436 -14.3027   True
    Batken      Naryn  -5.9067 0.8198 -17.5668   5.7534  False
    Batken        Osh -27.4294    0.0 -34.8806 -19.9781   True
    Batken   Osh_city -14.6447 0.0003 -24.8461  -4.4433   True
    Batken      Talas    -8.99 0.1336 -19.1665   1.1864  False
   Bishkek       Chui -19.7178    0.0 -27.4727 -11.9629   True
   Bishkek  Issyk-Kul -13.8162    0.0 -22.4269  -5.2056   True
   Bishkek Jalal-Abad -31.5869    0.0 -38.5281 -24.6456   True
   Bishkek      Naryn -16.1242 0.0005 -27.7087  -4.5398   True
   Bishkek        Osh -37.6469    0.0 -44.9793 -30.3145

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken,Bishkek,10.2175,0.0024,2.2178,18.2172,True
1,Batken,Chui,-9.5003,0.0057,-17.3677,-1.6329,True
2,Batken,Issyk-Kul,-3.5987,0.9366,-12.3109,5.1134,False
3,Batken,Jalal-Abad,-21.3693,0.0000,-28.4360,-14.3027,True
4,Batken,Naryn,-5.9067,0.8198,-17.5668,5.7534,False
5,Batken,Osh,-27.4294,0.0000,-34.8806,-19.9781,True
6,Batken,Osh_city,-14.6447,0.0003,-24.8461,-4.4433,True
7,Batken,Talas,-8.9900,0.1336,-19.1665,1.1864,False
8,Bishkek,Chui,-19.7178,0.0000,-27.4727,-11.9629,True
9,Bishkek,Issyk-Kul,-13.8162,0.0000,-22.4269,-5.2056,True


####Reading Comp Score - Regions - Tukey Test
Bishkek outperformed all other regions, significantly, on this subtask


###Silent Reading - Regions - Tukey Test

In [57]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['word_score_pcnt'], base['region'])

# Print results
print(tukey_results)
wordregtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
wordregtukey

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
  group1     group2   meandiff p-adj   lower    upper   reject
--------------------------------------------------------------
    Batken    Bishkek  -1.6873 0.9978  -8.5852   5.2106  False
    Batken       Chui   1.6998 0.9974  -5.0841   8.4836  False
    Batken  Issyk-Kul  -8.8441  0.008 -16.3563  -1.3318   True
    Batken Jalal-Abad  -6.8007 0.0158 -12.8941  -0.7073   True
    Batken      Naryn  -0.6885    1.0 -10.7426   9.3657  False
    Batken        Osh -16.2093    0.0 -22.6343  -9.7843   True
    Batken   Osh_city  -5.7992 0.5106 -14.5956   2.9972  False
    Batken      Talas   -2.695 0.9897 -11.4699   6.0799  False
   Bishkek       Chui   3.3871 0.8199  -3.2998  10.0739  False
   Bishkek  Issyk-Kul  -7.1568 0.0691 -14.5815    0.268  False
   Bishkek Jalal-Abad  -5.1134 0.1659 -11.0986   0.8718  False
   Bishkek      Naryn   0.9988    1.0  -8.9901  10.9878  False
   Bishkek        Osh  -14.522    0.0 -20.8445  -8.1995

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken,Bishkek,-1.6873,0.9978,-8.5852,5.2106,False
1,Batken,Chui,1.6998,0.9974,-5.0841,8.4836,False
2,Batken,Issyk-Kul,-8.8441,0.0080,-16.3563,-1.3318,True
3,Batken,Jalal-Abad,-6.8007,0.0158,-12.8941,-0.7073,True
4,Batken,Naryn,-0.6885,1.0000,-10.7426,9.3657,False
5,Batken,Osh,-16.2093,0.0000,-22.6343,-9.7843,True
6,Batken,Osh_city,-5.7992,0.5106,-14.5956,2.9972,False
7,Batken,Talas,-2.6950,0.9897,-11.4699,6.0799,False
8,Bishkek,Chui,3.3871,0.8199,-3.2998,10.0739,False
9,Bishkek,Issyk-Kul,-7.1568,0.0691,-14.5815,0.2680,False


####Silent reading -regions - tukey test
Now THIS is interesting.
1. Bishkek did *not* outperform in the silent reading subtask, as opposed to the others! Let's look into this a bit more
2. Every region did better, significantly, than Osh. However, the Uzbek students in Osh had *the highest* average scores on silent reading. If the sample was more aligned to the demographics in the region, i.e., more students took the test in uzbek, would this significance be the same?


###Invented Word Score - Regions - Tukey Test

In [58]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['invent_word_score_pcnt'], base['region'])

# Print results
print(tukey_results)
inventregtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
inventregtukey

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
  group1     group2   meandiff p-adj   lower    upper   reject
--------------------------------------------------------------
    Batken    Bishkek  12.9953    0.0   6.9114  19.0792   True
    Batken       Chui   0.6003    1.0   -5.383   6.5836  False
    Batken  Issyk-Kul  -1.2951 0.9996  -7.9208   5.3306  False
    Batken Jalal-Abad  -12.951    0.0 -18.3254  -7.5767   True
    Batken      Naryn  -1.7779 0.9995 -10.6456   7.0898  False
    Batken        Osh -14.8878    0.0 -20.5546   -9.221   True
    Batken   Osh_city  -1.7705 0.9987  -9.5289   5.9878  False
    Batken      Talas  -1.3249 0.9998  -9.0643   6.4145  False
   Bishkek       Chui  -12.395    0.0 -18.2927  -6.4972   True
   Bishkek  Issyk-Kul -14.2904    0.0  -20.839  -7.7418   True
   Bishkek Jalal-Abad -25.9463    0.0 -31.2252 -20.6674   True
   Bishkek      Naryn -14.7732    0.0 -23.5834   -5.963   True
   Bishkek        Osh -27.8831    0.0 -33.4595 -22.3067

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken,Bishkek,12.9953,0.0000,6.9114,19.0792,True
1,Batken,Chui,0.6003,1.0000,-5.3830,6.5836,False
2,Batken,Issyk-Kul,-1.2951,0.9996,-7.9208,5.3306,False
3,Batken,Jalal-Abad,-12.9510,0.0000,-18.3254,-7.5767,True
4,Batken,Naryn,-1.7779,0.9995,-10.6456,7.0898,False
5,Batken,Osh,-14.8878,0.0000,-20.5546,-9.2210,True
6,Batken,Osh_city,-1.7705,0.9987,-9.5289,5.9878,False
7,Batken,Talas,-1.3249,0.9998,-9.0643,6.4145,False
8,Bishkek,Chui,-12.3950,0.0000,-18.2927,-6.4972,True
9,Bishkek,Issyk-Kul,-14.2904,0.0000,-20.8390,-7.7418,True


#### Invented Word - Regions - Tukey Test
1. Back to Bishkek outperforming all other regions significantly
2. Osh being outperformed by other regions. Osh City outperforming Osh

###Listening Comp Score - Regions - Tukey Test

In [59]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['list_comp_score_pcnt'], base['region'])

# Print results
print(tukey_results)
listregtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
listregtukey

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
  group1     group2   meandiff p-adj   lower    upper   reject
--------------------------------------------------------------
    Batken    Bishkek   3.7997 0.8507  -4.0062  11.6056  False
    Batken       Chui   -4.997 0.5289 -12.6739   2.6798  False
    Batken  Issyk-Kul   0.0742    1.0  -8.4269   8.5754  False
    Batken Jalal-Abad -10.0997 0.0002 -16.9952  -3.2041   True
    Batken      Naryn   -6.558 0.6894 -17.9357   4.8197  False
    Batken        Osh -21.5413    0.0 -28.8121 -14.2705   True
    Batken   Osh_city  -8.6608 0.1478 -18.6152   1.2935  False
    Batken      Talas  -8.2425 0.1968 -18.1725   1.6875  False
   Bishkek       Chui  -8.7967 0.0095 -16.3638  -1.2297   True
   Bishkek  Issyk-Kul  -3.7255 0.9068 -12.1276   4.6767  False
   Bishkek Jalal-Abad -13.8994    0.0 -20.6725  -7.1263   True
   Bishkek      Naryn -10.3577 0.1032 -21.6616   0.9462  False
   Bishkek        Osh  -25.341    0.0 -32.4958 -18.1862

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken,Bishkek,3.7997,0.8507,-4.0062,11.6056,False
1,Batken,Chui,-4.9970,0.5289,-12.6739,2.6798,False
2,Batken,Issyk-Kul,0.0742,1.0000,-8.4269,8.5754,False
3,Batken,Jalal-Abad,-10.0997,0.0002,-16.9952,-3.2041,True
4,Batken,Naryn,-6.5580,0.6894,-17.9357,4.8197,False
5,Batken,Osh,-21.5413,0.0000,-28.8121,-14.2705,True
6,Batken,Osh_city,-8.6608,0.1478,-18.6152,1.2935,False
7,Batken,Talas,-8.2425,0.1968,-18.1725,1.6875,False
8,Bishkek,Chui,-8.7967,0.0095,-16.3638,-1.2297,True
9,Bishkek,Issyk-Kul,-3.7255,0.9068,-12.1276,4.6767,False


####Listening Comp -Regions-Tukey Test
1. Bishkek outperformed only half of the regions this time. Chui, Jalal-Abad, Osh, and Osh City

##ANOVA - one way test - Language within a region by subtask

###Oral Reading Score Score - between languages within a region

In [60]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('oral_read_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

ANOVA for Region: Batken
                    sum_sq     df          F        PR(>F)
C(language)   27057.548323    3.0  13.419231  3.091704e-08
Residual     199616.305498  297.0        NaN           NaN


ANOVA for Region: Bishkek
                    sum_sq     df          F    PR(>F)
C(language)   10392.732819    1.0  24.361557  0.000001
Residual     134806.801772  316.0        NaN       NaN


ANOVA for Region: Chui
                    sum_sq     df         F    PR(>F)
C(language)      83.601680    1.0  0.118575  0.730799
Residual     239013.759024  339.0       NaN       NaN


ANOVA for Region: Issyk-Kul
                    sum_sq     df         F    PR(>F)
C(language)    1710.704114    1.0  2.477204  0.116894
Residual     157451.943712  228.0       NaN       NaN


ANOVA for Region: Jalal-Abad
                    sum_sq     df         F    PR(>F)
C(language)   16005.042902    2.0  8.860669  0.000162
Residual     521118.067443  577.0       NaN       NaN


ANOVA for Region: Naryn
       

Takeaways of ANOVA test within regions for the oral reading  subtask score - did language matter within the region?

1. Batken - significant as p <= .05
2. Bishkek - significant as p <= .05
3. Chui - not significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - significant as p <= .05
9. Talas - not significant as p > .05


###Reading Comprehension - between languages within a region

In [61]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('read_comp_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

ANOVA for Region: Batken
                    sum_sq     df          F        PR(>F)
C(language)   98955.200191    3.0  38.820759  3.400542e-21
Residual     252353.769909  297.0        NaN           NaN


ANOVA for Region: Bishkek
                    sum_sq     df         F        PR(>F)
C(language)   20831.193483    1.0  26.74821  4.122955e-07
Residual     246097.108404  316.0       NaN           NaN


ANOVA for Region: Chui
                    sum_sq     df          F    PR(>F)
C(language)   13570.677133    1.0  13.644068  0.000257
Residual     337176.536943  339.0        NaN       NaN


ANOVA for Region: Issyk-Kul
                    sum_sq     df         F    PR(>F)
C(language)      36.469356    1.0  0.037748  0.846124
Residual     220278.313253  228.0       NaN       NaN


ANOVA for Region: Jalal-Abad
                    sum_sq     df         F    PR(>F)
C(language)   13102.068924    2.0  6.306486  0.001953
Residual     599374.482800  577.0       NaN       NaN


ANOVA for Region: N

Oral Reading Comprehension  subtask, significance of language within regions
1. Batken - significant as p <= .05
2. Bishkek -  significant as p >= .05
3. Chui -  significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - not significant as p >= .05
9. Talas - not significant as p > .05

###Silent Reading Comprehension - between languages within a region

In [62]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('word_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

ANOVA for Region: Batken
                    sum_sq     df          F        PR(>F)
C(language)   94895.942044    3.0  65.745064  1.251100e-32
Residual     142895.871910  297.0        NaN           NaN


ANOVA for Region: Bishkek
                    sum_sq     df         F    PR(>F)
C(language)    4653.912911    1.0  6.647983  0.010379
Residual     221215.433001  316.0       NaN       NaN


ANOVA for Region: Chui
                    sum_sq     df           F        PR(>F)
C(language)   72443.010357    1.0  116.787349  1.371596e-23
Residual     210281.171461  339.0         NaN           NaN


ANOVA for Region: Issyk-Kul
                    sum_sq     df         F    PR(>F)
C(language)     117.355436    1.0  0.146141  0.702606
Residual     183089.966303  228.0       NaN       NaN


ANOVA for Region: Jalal-Abad
                    sum_sq     df          F        PR(>F)
C(language)   20632.504730    2.0  15.301226  3.349587e-07
Residual     389019.652166  577.0        NaN           NaN


A

Word Score Percent subtask, significance of language within regions
1. Batken - significant as p <= .05
2. Bishkek - not significant as p >= .05
3. Chui -  significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - significant as p <= .05
9. Talas - not significant as p > .05

###Invented Word Score - between languages within a region

In [63]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('invent_word_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

ANOVA for Region: Batken
                    sum_sq     df        F    PR(>F)
C(language)    7494.583537    3.0  4.47693  0.004307
Residual     165730.479586  297.0      NaN       NaN


ANOVA for Region: Bishkek
                    sum_sq     df          F    PR(>F)
C(language)    8195.408011    1.0  16.791331  0.000053
Residual     154231.308970  316.0        NaN       NaN


ANOVA for Region: Chui
                    sum_sq     df         F    PR(>F)
C(language)     574.067017    1.0  0.890813  0.345929
Residual     218461.827411  339.0       NaN       NaN


ANOVA for Region: Issyk-Kul
                    sum_sq     df         F   PR(>F)
C(language)       7.357255    1.0  0.012344  0.91163
Residual     135886.903614  228.0       NaN      NaN


ANOVA for Region: Jalal-Abad
                    sum_sq     df         F    PR(>F)
C(language)   10205.082087    2.0  8.288309  0.000283
Residual     355219.159292  577.0       NaN       NaN


ANOVA for Region: Naryn
                   sum_sq   

Takeaways of ANOVA test within regions for the invented word subtask score - did language matter within the region?

1. Batken - significant as p <= .05
2. Bishkek - significant as p <= .05
3. Chui - not significant as p > .05
4. Issyk-Kul - not significant as p > .05
5. Jalal-Abad - significant as p <= .05
6. Naryn - not significant as p >= .05
7. Osh - significant as p <= .05
8. Osh City - significant as p <= .05
9. Talas - not significant as p > .05


###List Comp Score - between languages within a region

In [64]:
# List of regions
regions = base['region'].unique()

# Loop through regions and run ANOVA for each region
for region in regions:
    subset = base[base['region'] == region]
    model = ols('list_comp_score_pcnt ~ C(language)', data=subset).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(f"ANOVA for Region: {region}")
    print(anova_table)
    print("\n")

ANOVA for Region: Batken
                    sum_sq     df           F        PR(>F)
C(language)  166955.871637    3.0  113.566731  5.288427e-49
Residual     145541.138330  297.0         NaN           NaN


ANOVA for Region: Bishkek
                    sum_sq     df          F    PR(>F)
C(language)    9940.834999    1.0  16.274167  0.000069
Residual     193023.944875  316.0        NaN       NaN


ANOVA for Region: Chui
                    sum_sq     df          F        PR(>F)
C(language)   65526.596570    1.0  88.290836  8.617193e-19
Residual     251594.811055  339.0        NaN           NaN


ANOVA for Region: Issyk-Kul
                    sum_sq     df         F    PR(>F)
C(language)     267.365767    1.0  0.329287  0.566645
Residual     185125.677711  228.0       NaN       NaN


ANOVA for Region: Jalal-Abad
                    sum_sq     df           F        PR(>F)
C(language)  155007.103297    2.0  104.437859  1.948187e-39
Residual     428192.896703  577.0         NaN           N

##Tukey Test - One Way Significant Results Within Region

In [65]:
baselinemean = base.pivot_table(index=
 ['region', 'language'],
 values=[
     'oral_read_score_pcnt',
     'read_comp_score_pcnt',
     'word_score_pcnt',
     'invent_word_score_pcnt',
     'list_comp_score_pcnt'],
aggfunc='mean')
baselinemean.round(2)

invent_word_score_pcnt  list_comp_score_pcnt  \
region     language                                                 
Batken     Kyrgyz                     62.86                 59.86   
           Russian                    47.37                 26.32   
           Tajik                      56.33                 89.33   
           Uzbek                      60.04                 97.20   
Bishkek    Kyrgyz                     66.23                 67.13   
           Russian                    76.57                 78.52   
Chui       Kyrgyz                     58.87                 76.95   
           Russian                    61.50                 48.89   
Issyk-Kul  Kyrgyz                     57.98                 70.84   
           Russian                    58.38                 68.44   
Jalal-Abad Kyrgyz                     46.47                 63.86   
           Russian                    51.51                 36.23   
           Uzbek                      38.98                 83.45   
Naryn      Kyrgyz                     55.55                 61.25   
           Russian                    67.88                 75.00   
Osh        Kyrgyz                     43.06                 52.93   
           Russian                    44.25                 20.35   
           Uzbek                      55.35                 98.00   
Osh_city   Kyrgyz                     54.53                 81.05   
           Russian                    59.64                 56.18   
           Uzbek                      41.20                 82.00   
Talas      Kyrgyz                     57.28                 68.80   
           Russian                    60.00                 44.50   

                     oral_read_score_pcnt  read_comp_score_pcnt  \
region     language                                               
Batken     Kyrgyz                   78.71                 64.14   
           Russian                  54.21                 27.37   
           Tajik                    65.70                 41.33   
           Uzbek                    83.20                 82.80   
Bishkek    Kyrgyz                   78.66                 63.41   
           Russian                  90.30                 79.89   
Chui       Kyrgyz                   71.76                 58.88   
           Russian                  72.76                 46.11   
Issyk-Kul  Kyrgyz                   76.13                 59.64   
           Russian                  70.05                 58.75   
Jalal-Abad Kyrgyz                   58.68                 45.72   
           Russian                  61.77                 34.72   
           Uzbek                    46.75                 40.00   
Naryn      Kyrgyz                   70.55                 56.50   
           Russian                  83.12                 60.00   
Osh        Kyrgyz                   54.58                 37.95   
           Russian                  48.84                 18.95   
           Uzbek                    70.62                 66.00   
Osh_city   Kyrgyz                   69.84                 61.05   
           Russian                  71.86                 46.36   
           Uzbek                    45.60                 46.00   
Talas      Kyrgyz                   74.49                 56.00   
           Russian                  73.95                 49.00   

                     word_score_pcnt  
region     language                   
Batken     Kyrgyz              54.16  
           Russian             37.34  
           Tajik               72.23  
           Uzbek               87.72  
Bishkek    Kyrgyz              57.89  
           Russian             65.68  
Chui       Kyrgyz              78.37  
           Russian             48.86  
Issyk-Kul  Kyrgyz              54.92  
           Russian             56.52  
Jalal-Abad Kyrgyz              60.85  
           Russian             47.72  
           Uzbek               61.69  
Naryn      Kyrgyz              62.

Significance of Language *within* a region
1. Oral Reading: Batken, Bishkek, Jalal-Abad, Osh, Osh City
2. Reading Comprehension: Batken, Bishkek, Chui, Jalal-Abad, Osh, Osh City
3. Silent Reading: Batken, Bishkek, Chui, Jalal-Abad, Osh, Osh City, Talas
4. Invented Word: Batken, Bishkek, Jalal-Abad, Osh, Osh City
5. Listening Comp:Batken, Bishkek, Chui, Jalal-Abad, Osh, Osh City, Talas

In [66]:
# Create a combined category for region and language
base['Region_Language'] = base['region'] + "_" + base['language']


<ipython-input-66-136fc0f7eed0>:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



###Oral Reading Tukey Test, Language within region

In [67]:
oralreadonewayanova = ['Batken', 'Bishkek', 'Jalal-Abad', 'Osh', 'Osh_city']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in oralreadonewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['oral_read_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

Tukey HSD Results for Region: Batken
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1  group2 meandiff p-adj   lower    upper   reject
--------------------------------------------------------
 Kyrgyz Russian -24.5038    0.0 -36.7557 -12.2518   True
 Kyrgyz   Tajik -13.0143 0.0627   -26.49   0.4614  False
 Kyrgyz   Uzbek     4.49  0.567  -4.4703  13.4503  False
Russian   Tajik  11.4895 0.2685  -4.8694  27.8483  False
Russian   Uzbek  28.9938    0.0  16.0978  41.8897   True
  Tajik   Uzbek  17.5043 0.0078   3.4405  31.5681   True
--------------------------------------------------------


Tukey HSD Results for Region: Bishkek
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1  group2 meandiff p-adj lower   upper  reject
---------------------------------------------------
Kyrgyz Russian  11.6427   0.0 7.0016 16.2837   True
---------------------------------------------------


Tukey HSD Results for Region: Jalal-Abad
  Multiple Comparison of Means - Tukey HSD, FWER=0

####Oral Reading Tukey Test Analysis:

Oral Reading:
1. Batken - Kyrgyz did better, significantly so, in comparison to Russian (Does not align with hypothesis). Uzbek outperformed Russian and Tajik.

2. Bishkek - Russian did significantly better compared to Kyrgyz (aligns with hypothesis)

3. Jalal-Abad - Kyrgyz and Russian both outperformed Uzbek results. Russian did not outperform Kyrgyz

4. Osh - Uzbek outperformed Kyrgyz and Russian in Osh - which aligns with the regional demographics

5. Osh_City: Kyrgyz and Russian outperformed Uzbek. However due to sample size of Only 10 uzbeks in Osh_City,  not likely this is significant.

###Reading Comp Tukey Test - Language within region

In [68]:
readcomponewayanova = ['Batken', 'Bishkek', 'Chui','Jalal-Abad', 'Osh', 'Osh_city']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in readcomponewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['read_comp_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

Tukey HSD Results for Region: Batken
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1  group2 meandiff p-adj   lower    upper   reject
--------------------------------------------------------
 Kyrgyz Russian -36.7744    0.0 -50.5501 -22.9988   True
 Kyrgyz   Tajik -22.8095 0.0007 -37.9611   -7.658   True
 Kyrgyz   Uzbek  18.6528    0.0   8.5782  28.7275   True
Russian   Tajik  13.9649  0.205  -4.4284  32.3582  False
Russian   Uzbek  55.4273    0.0  40.9275   69.927   True
  Tajik   Uzbek  41.4624    0.0  25.6496  57.2751   True
--------------------------------------------------------


Tukey HSD Results for Region: Bishkek
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1  group2 meandiff p-adj  lower  upper  reject
---------------------------------------------------
Kyrgyz Russian  16.4833   0.0 10.2127 22.754   True
---------------------------------------------------


Tukey HSD Results for Region: Chui
 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
g

####Reading Comp Tukey Analysis

1. Batken: Russian did poor! Signifcant differnece in results with Kyrgyz and Uzbek outperforming Russian. (does not fit hypothesis!)
2. Bishkek, Russian outperformed Kyrgyz  (fits hypothesis)
3. Chui, Kyrgyz outperformed Russian. (does not fit hypothesis)
4. in Jalal-Abad - kyrgyz outperformed Russian significantly. average scores were however poor. No other significance found.
5. In Osh, Uzbek far outperformed Russian, as did Kyrgyz. significant differences found and students who selected Russian did substantially worse than their peers in osh on reading copmrehension.
6. No significance found within Osh city between languages.

###Silent Reading Tukey Test - Language within Region

In [69]:
wordscoreonewayanova = ['Batken', 'Bishkek', 'Chui','Jalal-Abad', 'Osh', 'Osh_city','Talas']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in wordscoreonewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['word_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

Tukey HSD Results for Region: Batken
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1  group2 meandiff p-adj   lower    upper  reject
-------------------------------------------------------
 Kyrgyz Russian -16.8222 0.0002 -27.1883  -6.456   True
 Kyrgyz   Tajik   18.069 0.0003   6.6675 29.4706   True
 Kyrgyz   Uzbek  33.5561    0.0   25.975 41.1373   True
Russian   Tajik  34.8912    0.0  21.0503 48.7321   True
Russian   Uzbek  50.3783    0.0  39.4673 61.2893   True
  Tajik   Uzbek  15.4871 0.0048    3.588 27.3862   True
-------------------------------------------------------


Tukey HSD Results for Region: Bishkek
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1  group2 meandiff p-adj  lower   upper  reject
----------------------------------------------------
Kyrgyz Russian   7.7911 0.0104 1.8459 13.7363   True
----------------------------------------------------


Tukey HSD Results for Region: Chui
 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1

####Silent Reading Tukey Analysis

1. Batken: Significance between all languages, with Kyrgyz, Tajik, and Uzbek outperforming Russian.  Interestingly, the minority languages (uzbek and tajik) outperformed Kyrgyz in this region!
2. Bishkek: Russian outperformed Kyrgyz significantly
3. Chui: Kyrgyz outperformed Russian significantly
4. Jalal-Abad: Kyrgyz and Uzbek outperformed Russian significantly
5. Osh Kyrgyz outperformed Russian. Uzbek outperformed Kyrgyz + Russian
6. Osh City - Kyrgyz outperformed Russian, no other significance found (osh city simple size for uzbek isnt helping! :-( )
7. Talas - Kyrgyz outperformed Russian


###Invented Word Score - Language within Region

In [70]:
inventwordoneway = ['Batken', 'Bishkek','Jalal-Abad', 'Osh', 'Osh_city']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in inventwordoneway:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['invent_word_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

Tukey HSD Results for Region: Batken
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1  group2 meandiff p-adj   lower    upper  reject
-------------------------------------------------------
 Kyrgyz Russian -15.4887 0.0022 -26.6524  -4.325   True
 Kyrgyz   Tajik  -6.5238 0.5175 -18.8026  5.7549  False
 Kyrgyz   Uzbek  -2.8141 0.8098 -10.9786  5.3503  False
Russian   Tajik   8.9649 0.4068  -5.9409 23.8707  False
Russian   Uzbek  12.6746 0.0288   0.9241 24.4251   True
  Tajik   Uzbek   3.7097 0.8775  -9.1049 16.5243  False
-------------------------------------------------------


Tukey HSD Results for Region: Bishkek
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1  group2 meandiff p-adj  lower  upper  reject
---------------------------------------------------
Kyrgyz Russian  10.3389 0.0001 5.3747 15.303   True
---------------------------------------------------


Tukey HSD Results for Region: Jalal-Abad
  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 gro

####Invented Word Score Tukey Analysis
1. Batken: Kyrgyz outperformed Russian, and uzbek outperformed Russian, significantly
2. Bishkek: Russian outperformed kyrgyz, significance
3. Jalal-Abad: Kyrgyz and Russian both significantly outperformed Uzbek
4. Osh - Kyrgyz and Russian were both outperformed by Uzbek
5. Osh City - nothing of significance to report due to small size of Uzbek population smapled

###Listening Comprehension Score - Language within Region

In [71]:
listcomponewayanova = ['Batken', 'Bishkek', 'Chui','Jalal-Abad', 'Osh', 'Osh_city','Talas']

# Loop through each region and perform Tukey HSD test for language comparisons within each region
for region in listcomponewayanova:
    # Filter the data for the current region
    subset = base[base['region'] == region]

    # Run Tukey HSD Test within the region for language differences
    tukey = pairwise_tukeyhsd(subset['list_comp_score_pcnt'], subset['language'], alpha=0.05)

    # Print the results for each region
    print(f"Tukey HSD Results for Region: {region}")
    print(tukey.summary())
    print("\n")

Tukey HSD Results for Region: Batken
 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
 group1  group2 meandiff p-adj  lower   upper   reject
------------------------------------------------------
 Kyrgyz Russian -33.5414   0.0 -44.003 -23.0797   True
 Kyrgyz   Tajik  29.4762   0.0 17.9696  40.9828   True
 Kyrgyz   Uzbek  37.3472   0.0 29.6962  44.9982   True
Russian   Tajik  63.0175   0.0 49.0491   76.986   True
Russian   Uzbek  70.8885   0.0  59.877  81.9001   True
  Tajik   Uzbek    7.871 0.329 -4.1377  19.8797  False
------------------------------------------------------


Tukey HSD Results for Region: Bishkek
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1  group2 meandiff p-adj  lower   upper  reject
----------------------------------------------------
Kyrgyz Russian  11.3867 0.0001 5.8333 16.9402   True
----------------------------------------------------


Tukey HSD Results for Region: Chui
 Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1  group2 m

####Listening Comp Score - Tukey Analysis
1. Batken: Kyrgyz outperformed Russian, but Tajik and Uzbek completely outperformed Kyrgyz and russian
2. Bishkek outperformed in RUssian compared to Kyrgyz
3. Kyrgyz outperformed Russian significantly
4. Jalal-Abad: Kyrgyz outperformed Russian, Uzbek outperformed Russian, UZbek outperformed Kyrgyz
5. Osh: Kyrgyz outperformed Russian, Uzbek outperformed Kyrgyz and Russian
6. Osh_City - Kyrgyz outperformed Russian, significant. Uzbek sample size is small but outperformed Russian significantly.
7. Talas: Kyrgyz outperformed russian significantly.

##Tukey Test - Two Way Group Pairings - Region Language Pairs

###Oral Reading Score Tukey Test

In [72]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['oral_read_score_pcnt'], base['Region_Language'])

# Print results
print(tukey_results)
oraltukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
oraltukey

             Multiple Comparison of Means - Tukey HSD, FWER=0.05              
      group1             group2       meandiff p-adj   lower    upper   reject
------------------------------------------------------------------------------
     Batken_Kyrgyz     Batken_Russian -24.5038 0.0002 -42.4541  -6.5534   True
     Batken_Kyrgyz       Batken_Tajik -13.0143 0.7398 -32.7575   6.7289  False
     Batken_Kyrgyz       Batken_Uzbek     4.49 0.9999  -8.6377  17.6178  False
     Batken_Kyrgyz     Bishkek_Kyrgyz  -0.0554    1.0  -12.032  11.9213  False
     Batken_Kyrgyz    Bishkek_Russian  11.5873 0.0239   0.6447  22.5299   True
     Batken_Kyrgyz        Chui_Kyrgyz  -6.9529 0.7847 -17.8005   3.8948  False
     Batken_Kyrgyz       Chui_Russian  -5.9504 0.9726 -17.5979   5.6971  False
     Batken_Kyrgyz   Issyk-Kul_Kyrgyz  -2.5818    1.0 -13.8423   8.6788  False
     Batken_Kyrgyz  Issyk-Kul_Russian  -8.6674 0.8945 -23.4748     6.14  False
     Batken_Kyrgyz  Jalal-Abad_Kyrgyz -20.0358    0.

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-24.5038,0.0002,-42.4541,-6.5534,True
1,Batken_Kyrgyz,Batken_Tajik,-13.0143,0.7398,-32.7575,6.7289,False
2,Batken_Kyrgyz,Batken_Uzbek,4.4900,0.9999,-8.6377,17.6178,False
3,Batken_Kyrgyz,Bishkek_Kyrgyz,-0.0554,1.0000,-12.0320,11.9213,False
4,Batken_Kyrgyz,Bishkek_Russian,11.5873,0.0239,0.6447,22.5299,True
...,...,...,...,...,...,...,...
248,Osh_city_Russian,Talas_Kyrgyz,2.6264,1.0000,-10.9328,16.1855,False
249,Osh_city_Russian,Talas_Russian,2.0864,1.0000,-16.0328,20.2055,False
250,Osh_city_Uzbek,Talas_Kyrgyz,28.8900,0.1670,-3.6573,61.4373,False
251,Osh_city_Uzbek,Talas_Russian,28.3500,0.3062,-6.3455,63.0455,False


###Reading Comprehension Score Tukey Test

In [73]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['read_comp_score_pcnt'], base['Region_Language'])

# Print results
readcomptukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])
readcomptukey

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-36.7744,0.0000,-57.1362,-16.4127,True
1,Batken_Kyrgyz,Batken_Tajik,-22.8095,0.0400,-45.2050,-0.4141,True
2,Batken_Kyrgyz,Batken_Uzbek,18.6528,0.0014,3.7615,33.5441,True
3,Batken_Kyrgyz,Bishkek_Kyrgyz,-0.7320,1.0000,-14.3176,12.8536,False
4,Batken_Kyrgyz,Bishkek_Russian,15.7513,0.0010,3.3387,28.1639,True
...,...,...,...,...,...,...,...
248,Osh_city_Russian,Talas_Kyrgyz,9.6364,0.8174,-5.7442,25.0170,False
249,Osh_city_Russian,Talas_Russian,2.6364,1.0000,-17.9168,23.1896,False
250,Osh_city_Uzbek,Talas_Kyrgyz,10.0000,1.0000,-26.9196,46.9196,False
251,Osh_city_Uzbek,Talas_Russian,3.0000,1.0000,-36.3564,42.3564,False


###Silent Reading Score Tukey Test

In [74]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['word_score_pcnt'], base['Region_Language'])
wordscoretukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Print results
print(tukey_results)
wordscoretukey

             Multiple Comparison of Means - Tukey HSD, FWER=0.05              
      group1             group2       meandiff p-adj   lower    upper   reject
------------------------------------------------------------------------------
     Batken_Kyrgyz     Batken_Russian -16.8222 0.0531 -33.7312   0.0868  False
     Batken_Kyrgyz       Batken_Tajik   18.069 0.0696  -0.5288  36.6669  False
     Batken_Kyrgyz       Batken_Uzbek  33.5561    0.0  21.1899  45.9223   True
     Batken_Kyrgyz     Bishkek_Kyrgyz   3.7272 0.9999  -7.5547  15.0091  False
     Batken_Kyrgyz    Bishkek_Russian  11.5183 0.0108   1.2104  21.8261   True
     Batken_Kyrgyz        Chui_Kyrgyz  24.2063    0.0  13.9879  34.4246   True
     Batken_Kyrgyz       Chui_Russian  -5.3032 0.9856  -16.275   5.6686  False
     Batken_Kyrgyz   Issyk-Kul_Kyrgyz   0.7574    1.0  -9.8499  11.3647  False
     Batken_Kyrgyz  Issyk-Kul_Russian   2.3513    1.0 -11.5971  16.2998  False
     Batken_Kyrgyz  Jalal-Abad_Kyrgyz   6.6846 0.598

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-16.8222,0.0531,-33.7312,0.0868,False
1,Batken_Kyrgyz,Batken_Tajik,18.0690,0.0696,-0.5288,36.6669,False
2,Batken_Kyrgyz,Batken_Uzbek,33.5561,0.0000,21.1899,45.9223,True
3,Batken_Kyrgyz,Bishkek_Kyrgyz,3.7272,0.9999,-7.5547,15.0091,False
4,Batken_Kyrgyz,Bishkek_Russian,11.5183,0.0108,1.2104,21.8261,True
...,...,...,...,...,...,...,...
248,Osh_city_Russian,Talas_Kyrgyz,9.4818,0.5058,-3.2907,22.2544,False
249,Osh_city_Russian,Talas_Russian,-2.0182,1.0000,-19.0862,15.0498,False
250,Osh_city_Uzbek,Talas_Kyrgyz,9.7000,1.0000,-20.9592,40.3592,False
251,Osh_city_Uzbek,Talas_Russian,-1.8000,1.0000,-34.4828,30.8828,False


###Invented Word Score Tukey Test

In [75]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['invent_word_score_pcnt'], base['Region_Language'])
inventtukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Print results
print(tukey_results)
inventtukey

             Multiple Comparison of Means - Tukey HSD, FWER=0.05              
      group1             group2       meandiff p-adj   lower    upper   reject
------------------------------------------------------------------------------
     Batken_Kyrgyz     Batken_Russian -15.4887 0.0715 -31.4691   0.4917  False
     Batken_Kyrgyz       Batken_Tajik  -6.5238 0.9996 -24.1004  11.0527  False
     Batken_Kyrgyz       Batken_Uzbek  -2.8141    1.0 -14.5012    8.873  False
     Batken_Kyrgyz     Bishkek_Kyrgyz   3.3754    1.0  -7.2869  14.0377  False
     Batken_Kyrgyz    Bishkek_Russian  13.7143 0.0001   3.9725   23.456   True
     Batken_Kyrgyz        Chui_Kyrgyz   -3.984 0.9982 -13.6413   5.6732  False
     Batken_Kyrgyz       Chui_Russian  -1.3571    1.0 -11.7264   9.0121  False
     Batken_Kyrgyz   Issyk-Kul_Kyrgyz  -4.8812 0.9842 -14.9061   5.1436  False
     Batken_Kyrgyz  Issyk-Kul_Russian  -4.4821 0.9999 -17.6646   8.7003  False
     Batken_Kyrgyz  Jalal-Abad_Kyrgyz -16.3877    0.

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-15.4887,0.0715,-31.4691,0.4917,False
1,Batken_Kyrgyz,Batken_Tajik,-6.5238,0.9996,-24.1004,11.0527,False
2,Batken_Kyrgyz,Batken_Uzbek,-2.8141,1.0000,-14.5012,8.8730,False
3,Batken_Kyrgyz,Bishkek_Kyrgyz,3.3754,1.0000,-7.2869,14.0377,False
4,Batken_Kyrgyz,Bishkek_Russian,13.7143,0.0001,3.9725,23.4560,True
...,...,...,...,...,...,...,...
248,Osh_city_Russian,Talas_Kyrgyz,-2.3564,1.0000,-14.4275,9.7147,False
249,Osh_city_Russian,Talas_Russian,0.3636,1.0000,-15.7671,16.4943,False
250,Osh_city_Uzbek,Talas_Kyrgyz,16.0800,0.9354,-12.8954,45.0554,False
251,Osh_city_Uzbek,Talas_Russian,18.8000,0.8539,-12.0879,49.6879,False


###List Comprehension Score Tukey Test

In [76]:
# Run Tukey's test
tukey_results = pairwise_tukeyhsd(base['list_comp_score_pcnt'], base['Region_Language'])
listcomptukey = pd.DataFrame(data=tukey_results._results_table.data[1:], columns=tukey_results._results_table.data[0])

# Print results
print(tukey_results)
listcomptukey

             Multiple Comparison of Means - Tukey HSD, FWER=0.05              
      group1             group2       meandiff p-adj   lower    upper   reject
------------------------------------------------------------------------------
     Batken_Kyrgyz     Batken_Russian -33.5414    0.0 -51.5349 -15.5478   True
     Batken_Kyrgyz       Batken_Tajik  29.4762    0.0   9.6854  49.2669   True
     Batken_Kyrgyz       Batken_Uzbek  37.3472    0.0  24.1878  50.5065   True
     Batken_Kyrgyz     Bishkek_Kyrgyz   7.2746  0.859  -4.7308  19.2801  False
     Batken_Kyrgyz    Bishkek_Russian  18.6614    0.0   7.6924  29.6303   True
     Batken_Kyrgyz        Chui_Kyrgyz  17.0972    0.0   6.2234  27.9709   True
     Batken_Kyrgyz       Chui_Russian -10.9683 0.0992 -22.6438   0.7073  False
     Batken_Kyrgyz   Issyk-Kul_Kyrgyz  10.9862 0.0683  -0.3015  22.2739  False
     Batken_Kyrgyz  Issyk-Kul_Russian   8.5804 0.9055  -6.2627  23.4234  False
     Batken_Kyrgyz  Jalal-Abad_Kyrgyz   4.0014 0.998

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-33.5414,0.0000,-51.5349,-15.5478,True
1,Batken_Kyrgyz,Batken_Tajik,29.4762,0.0000,9.6854,49.2669,True
2,Batken_Kyrgyz,Batken_Uzbek,37.3472,0.0000,24.1878,50.5065,True
3,Batken_Kyrgyz,Bishkek_Kyrgyz,7.2746,0.8590,-4.7308,19.2801,False
4,Batken_Kyrgyz,Bishkek_Russian,18.6614,0.0000,7.6924,29.6303,True
...,...,...,...,...,...,...,...
248,Osh_city_Russian,Talas_Kyrgyz,12.6182,0.1114,-0.9736,26.2099,False
249,Osh_city_Russian,Talas_Russian,-11.6818,0.7794,-29.8446,6.4809,False
250,Osh_city_Uzbek,Talas_Kyrgyz,-13.2000,0.9986,-45.8256,19.4256,False
251,Osh_city_Uzbek,Talas_Russian,-37.5000,0.0185,-72.2790,-2.7210,True


##Tukey Analysis

The hypothesis was that Russian would, by and large, outperform the other languages, especially in Bishkek.

For that reason, all significant differences in any pairing with "Russian" and "Bishkek" will be reviewed.

###Original sample count by region and language, and pivot table with means by subtask

In [77]:
baselinecount = base.groupby(by=['region','language']).size().rename('count')
baselinecount
regethnics

region      language
Batken      Kyrgyz      140
            Russian      38
            Tajik        30
            Uzbek        93
Bishkek     Kyrgyz      129
            Russian     189
Chui        Kyrgyz      197
            Russian     144
Issyk-Kul   Kyrgyz      166
            Russian      64
Jalal-Abad  Kyrgyz      311
            Russian     159
            Uzbek       110
Naryn       Kyrgyz       80
            Russian      16
Osh         Kyrgyz      283
            Russian     114
            Uzbek        40
Osh_city    Kyrgyz       19
            Russian     110
            Uzbek        10
Talas       Kyrgyz      100
            Russian      40
Name: count, dtype: int64

,Ethnicity,Total Population,Batken,Jalal-Abad,Issyk-Kul,Naryn,Osh,Talas,Chui,Bishkek,Osh city
0,Sum population,"7,037,590","570,898","1,311,007","538,384","308,348","1,460,425","273,509","1,068,702","1,145,044","361,273"
2,Kyrgyz,"5,470,806","451,422","967,355","492,452","307,095","1,003,764","260,038","798,347","975,128","215,205"
3,Russians,"277,646","2,021","5,062","27,424",88,"1,254","3,257","124,640","110,439","3,461"
4,Uzbeks,"995,454","78,314","321,461","3,463",291,"421,519","1,006","17,480","15,899","136,021"
11,Tajiks,"60,752","36,921","7,153",213,23,"8,626",53,"5,208","1,743",812


In [78]:
baselinemean = base.pivot_table(index=
 ['region', 'language'],
 values=[
     'oral_read_score_pcnt',
     'read_comp_score_pcnt',
     'word_score_pcnt',
     'invent_word_score_pcnt',
     'list_comp_score_pcnt'],
aggfunc='mean')
baselinemean.round(2)

invent_word_score_pcnt  list_comp_score_pcnt  \
region     language                                                 
Batken     Kyrgyz                     62.86                 59.86   
           Russian                    47.37                 26.32   
           Tajik                      56.33                 89.33   
           Uzbek                      60.04                 97.20   
Bishkek    Kyrgyz                     66.23                 67.13   
           Russian                    76.57                 78.52   
Chui       Kyrgyz                     58.87                 76.95   
           Russian                    61.50                 48.89   
Issyk-Kul  Kyrgyz                     57.98                 70.84   
           Russian                    58.38                 68.44   
Jalal-Abad Kyrgyz                     46.47                 63.86   
           Russian                    51.51                 36.23   
           Uzbek                      38.98                 83.45   
Naryn      Kyrgyz                     55.55                 61.25   
           Russian                    67.88                 75.00   
Osh        Kyrgyz                     43.06                 52.93   
           Russian                    44.25                 20.35   
           Uzbek                      55.35                 98.00   
Osh_city   Kyrgyz                     54.53                 81.05   
           Russian                    59.64                 56.18   
           Uzbek                      41.20                 82.00   
Talas      Kyrgyz                     57.28                 68.80   
           Russian                    60.00                 44.50   

                     oral_read_score_pcnt  read_comp_score_pcnt  \
region     language                                               
Batken     Kyrgyz                   78.71                 64.14   
           Russian                  54.21                 27.37   
           Tajik                    65.70                 41.33   
           Uzbek                    83.20                 82.80   
Bishkek    Kyrgyz                   78.66                 63.41   
           Russian                  90.30                 79.89   
Chui       Kyrgyz                   71.76                 58.88   
           Russian                  72.76                 46.11   
Issyk-Kul  Kyrgyz                   76.13                 59.64   
           Russian                  70.05                 58.75   
Jalal-Abad Kyrgyz                   58.68                 45.72   
           Russian                  61.77                 34.72   
           Uzbek                    46.75                 40.00   
Naryn      Kyrgyz                   70.55                 56.50   
           Russian                  83.12                 60.00   
Osh        Kyrgyz                   54.58                 37.95   
           Russian                  48.84                 18.95   
           Uzbek                    70.62                 66.00   
Osh_city   Kyrgyz                   69.84                 61.05   
           Russian                  71.86                 46.36   
           Uzbek                    45.60                 46.00   
Talas      Kyrgyz                   74.49                 56.00   
           Russian                  73.95                 49.00   

                     word_score_pcnt  
region     language                   
Batken     Kyrgyz              54.16  
           Russian             37.34  
           Tajik               72.23  
           Uzbek               87.72  
Bishkek    Kyrgyz              57.89  
           Russian             65.68  
Chui       Kyrgyz              78.37  
           Russian             48.86  
Issyk-Kul  Kyrgyz              54.92  
           Russian             56.52  
Jalal-Abad Kyrgyz              60.85  
           Russian             47.72  
           Uzbek               61.69  
Naryn      Kyrgyz              62.

###Tukey Analysis - Two Way

In [79]:
oraltukey[oraltukey['reject'] == True]

,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-24.5038,0.0002,-42.4541,-6.5534,True
4,Batken_Kyrgyz,Bishkek_Russian,11.5873,0.0239,0.6447,22.5299,True
9,Batken_Kyrgyz,Jalal-Abad_Kyrgyz,-20.0358,0.0000,-30.0235,-10.0482,True
10,Batken_Kyrgyz,Jalal-Abad_Russian,-16.9407,0.0000,-28.3141,-5.5673,True
11,Batken_Kyrgyz,Jalal-Abad_Uzbek,-31.9597,0.0000,-44.4631,-19.4564,True
...,...,...,...,...,...,...,...
231,Osh_Kyrgyz,Talas_Russian,19.3670,0.0051,2.7903,35.9436,True
232,Osh_Russian,Osh_Uzbek,21.7829,0.0028,3.7487,39.8171,True
234,Osh_Russian,Osh_city_Russian,23.0215,0.0000,9.9058,36.1373,True
236,Osh_Russian,Talas_Kyrgyz,25.6479,0.0000,12.2025,39.0933,True


In [80]:
readcomptukey[readcomptukey['reject'] == True]


,group1,group2,meandiff,p-adj,lower,upper,reject
0,Batken_Kyrgyz,Batken_Russian,-36.7744,0.0000,-57.1362,-16.4127,True
1,Batken_Kyrgyz,Batken_Tajik,-22.8095,0.0400,-45.2050,-0.4141,True
2,Batken_Kyrgyz,Batken_Uzbek,18.6528,0.0014,3.7615,33.5441,True
4,Batken_Kyrgyz,Bishkek_Russian,15.7513,0.0010,3.3387,28.1639,True
6,Batken_Kyrgyz,Chui_Russian,-18.0317,0.0002,-31.2439,-4.8196,True
...,...,...,...,...,...,...,...
232,Osh_Russian,Osh_Uzbek,47.0526,0.0000,26.5958,67.5095,True
233,Osh_Russian,Osh_city_Kyrgyz,42.1053,0.0000,14.5213,69.6892,True
234,Osh_Russian,Osh_city_Russian,27.4163,0.0000,12.5386,42.2940,True
236,Osh_Russian,Talas_Kyrgyz,37.0526,0.0000,21.8011,52.3042,True


In [81]:
wordscoretukey[wordscoretukey['reject'] == True]


,group1,group2,meandiff,p-adj,lower,upper,reject
2,Batken_Kyrgyz,Batken_Uzbek,33.5561,0.0000,21.1899,45.9223,True
4,Batken_Kyrgyz,Bishkek_Russian,11.5183,0.0108,1.2104,21.8261,True
5,Batken_Kyrgyz,Chui_Kyrgyz,24.2063,0.0000,13.9879,34.4246,True
15,Batken_Kyrgyz,Osh_Russian,-20.7695,0.0000,-32.4313,-9.1078,True
16,Batken_Kyrgyz,Osh_Uzbek,34.1357,0.0000,17.5625,50.7089,True
...,...,...,...,...,...,...,...
237,Osh_Russian,Talas_Russian,19.9053,0.0049,2.9173,36.8932,True
239,Osh_Uzbek,Osh_city_Russian,-32.9818,0.0000,-50.0498,-15.9138,True
240,Osh_Uzbek,Osh_city_Uzbek,-33.2000,0.0413,-65.8828,-0.5172,True
241,Osh_Uzbek,Talas_Kyrgyz,-23.5000,0.0002,-40.7941,-6.2059,True


In [82]:
inventtukey[inventtukey['reject'] == True]


,group1,group2,meandiff,p-adj,lower,upper,reject
4,Batken_Kyrgyz,Bishkek_Russian,13.7143,0.0001,3.9725,23.4560,True
9,Batken_Kyrgyz,Jalal-Abad_Kyrgyz,-16.3877,0.0000,-25.2792,-7.4961,True
10,Batken_Kyrgyz,Jalal-Abad_Russian,-11.3477,0.0103,-21.4730,-1.2224,True
11,Batken_Kyrgyz,Jalal-Abad_Uzbek,-23.8753,0.0000,-35.0066,-12.7441,True
14,Batken_Kyrgyz,Osh_Kyrgyz,-19.7971,0.0000,-28.8241,-10.7700,True
...,...,...,...,...,...,...,...
228,Osh_Kyrgyz,Osh_city_Russian,16.5763,0.0000,6.7602,26.3924,True
230,Osh_Kyrgyz,Talas_Kyrgyz,14.2199,0.0001,4.0565,24.3833,True
231,Osh_Kyrgyz,Talas_Russian,16.9399,0.0069,2.1825,31.6974,True
234,Osh_Russian,Osh_city_Russian,15.3907,0.0005,3.7144,27.0671,True


# New Section